
# Schematic Outline for Data Preprocessing in Speech-to-Text (SST) Model
## 1. Audio Data Preparation

###**CODE TO BE PERFORMED IN JUPYTER LAB FOR IMPROVING THE PERFORMANCE**

1. Collection of Audio Files
* Action: Gather all audio files from the designated source directory.
* Format: MP3 initially, to be converted to WAV.
2. Conversion from MP3 to WAV
* Purpose: Convert audio files from MP3 to WAV format for better compatibility with preprocessing tools and models.
* Tools: Utilize libraries like pydub or command-line tools like ffmpeg.
3. Noise Reduction
* Action: Apply noise reduction techniques to minimize background noise and enhance audio clarity.
* Tools: Libraries such as noisereduce or functions within librosa.
4. Trimming Silence
* Action: Remove leading and trailing silences from audio files to focus on the spoken content.
* Tools: Use librosa.effects.trim or similar functions.
5. Normalization of Audio Levels
* Action: Normalize audio amplitudes to ensure consistent volume levels across all samples.
* Tools: Functions like librosa.util.normalize.
6. Resampling
* Action: Resample audio files to a standard sampling rate (e.g., 16,000 Hz) to maintain uniformity.
* Tools: librosa.resample or equivalent.
7. Padding to Uniform Length
* Action: Pad audio sequences with zeros or truncate them to achieve a fixed length (max_length), facilitating batch processing.
* Purpose: Ensures all audio inputs have the same dimensions for model training.
* Tools: numpy.pad or TensorFlow/Keras padding utilities.
# 2. Text Data Preparation
  1. Loading and Reading the TSV File
* Action: Load the transcription data from a TSV (Tab-Separated Values) file using pandas or similar libraries.
* Content: Should include columns like audio_filename, transcription, and language.
#  3. Text Cleaning
Actions:
  *   Eliminate Uppercase Letters: Convert all text to lowercase to maintain consistency.
  *   Remove Unwanted Symbols and Characters: Use regular expressions (re) to strip out non-alphanumeric characters except necessary punctuation.
  *   Standardize Punctuation: Ensure consistent use of punctuation marks.

  3. Phonetic Transcription of Text
* Action: Convert cleaned text into its phonetic representation to aid the model in understanding pronunciation.
* Purpose: Enhances the model's ability to map audio features to textual output accurately.
* Tools: Libraries like phonemizer, pronouncing, or APIs that provide phonetic transcriptions.
  4. Adding Language Tags
* Action: Prefix each transcription with a language-specific token (e.g., <ITALIANO>, <ESPAÑOL>, <SWAHILI>).
* Purpose: Enables the model to recognize and handle multiple languages within the dataset effectively.
  5. Converting TSV to JSON
* Action: Transform the cleaned and augmented TSV data into a JSON format containing:
  * Audio Filename: Reference to the corresponding audio file.
  * Cleaned Text: The standardized transcription.
  * Phonetic Transcription: Phonetic representation of the text.
  * Language Tag: Indicator of the language used.
* Tools: Utilize pandas for data manipulation and Python's json module for conversion.
# 3. Alignment of Text and Audio
  1. Verification of Correspondence
Action: Ensure that each audio file has a corresponding transcription and vice versa.
Purpose: Prevents mismatches that can lead to training errors.
  2. Temporal Alignment (Optional)
Action: Align the transcription text temporally with specific segments of the audio if precise synchronization is required.
Tools: Tools like Montreal Forced Aligner or gentle can be used for accurate alignment.
# 4. Tokenization and Padding of Text
  1. Creating and Fitting the Tokenizer
Action: Use Keras's Tokenizer to convert text into numerical sequences.
Configuration:
char_level=True for character-level tokenization.
Disable filters or adjust based on preprocessing needs.
Purpose: Transforms textual data into a format suitable for model ingestion.
  2. Tokenizing the Transcriptions
Action: Apply the tokenizer to convert each transcription into a sequence of integers representing characters or words.
  3. Applying Padding and Truncation
Action: Use pad_sequences from Keras to pad or truncate tokenized sequences to a fixed max_length.
Purpose: Ensures all text inputs have uniform length for efficient batch processing during training.
# 5. Splitting Data into Training and Validation Sets
  1. Data Partitioning
Action: Divide the preprocessed data into training and validation sets, typically using an 80/20 split.
Tools: Utilize train_test_split from sklearn.model_selection.
  2. Ensuring Balanced Distribution
Action: Verify that the distribution of languages and other relevant attributes is balanced across both sets to prevent bias.
# 6. Creation of Custom Data Generators
  1. Defining Generators for Training and Validation
Action: Implement generators that:
Handle batching of data.
Shuffle data to ensure randomness in each epoch.
Apply padding and truncation dynamically based on max_length.
Purpose: Facilitates efficient and scalable data feeding into the model during training.
  2. Optimizing Generator Performance
Action: Ensure that generators are optimized for performance, possibly by leveraging TensorFlow's tf.data.Dataset for enhanced scalability and speed.

transformers==4.18.0
librosa==0.8.1
tensorboard==2.7.0
jiwer==2.3.0
pydub==0.25.1
epitran==1.3.4
pandas==1.3.5
sentencepiece==0.1.96
tqdm==4.62.3


In [ ]:
!pip install transformers
!pip install librosa
!pip install tensorboard
!pip install jiwer
!pip install pydub
!apt-get install ffmpeg -y
!pip install epitran
!pip install pandas
!pip install sentencepiece tqdm
!pip install g2p_en




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.4 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.1/184.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.1/577.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.6/488.6 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 57.6 MB/s eta 0:00:00
  Created wheel for unicodecsv: filename=unicodecsv-0.14.1-py3-none-any.whl size=10744 sha256=4178180639ce1e56391be6aaff16109dc7e8da5d3e818f66fcee2652d100d219
  Stored in direct

Cargar y Preprocesar los Datos
Vamos a convertir los archivos MP3 a WAV, cargar las transcripciones desde el archivo validated.tsv y preprocesar los datos.

In [ ]:
# Step 1: Preprocessing - Connect to Google Drive

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


Convert MP3 to WAV
Convert all MP3 audio files to WAV format for consistency and compatibility with processing libraries.

In [ ]:
import os

italian_audio_dir = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/audio"

# Check if the directory exists
if os.path.exists(italian_audio_dir):
    print(f"The directory exists: {italian_audio_dir}")
    # List the contents
    print("Contents of the 'audio' directory:")
    print(os.listdir(italian_audio_dir))
else:
    print(f"The directory does NOT exist: {italian_audio_dir}")


The directory exists: /content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/audio
Contents of the 'audio' directory:
['common_voice_it_20283956.mp3', 'common_voice_it_20283957.mp3', 'common_voice_it_20283958.mp3', 'common_voice_it_20283959.mp3', 'common_voice_it_20283960.mp3', 'common_voice_it_25331358.mp3', 'common_voice_it_25331359.mp3', 'common_voice_it_25331360.mp3', 'common_voice_it_25331361.mp3', 'common_voice_it_25331362.mp3', 'common_voice_it_20079235.mp3', 'common_voice_it_20079234.mp3', 'common_voice_it_20079236.mp3', 'common_voice_it_20079237.mp3', 'common_voice_it_20079238.mp3', 'common_voice_it_20085158.mp3', 'common_voice_it_20085159.mp3', 'common_voice_it_20085160.mp3', 'common_voice_it_20085161.mp3', 'common_voice_it_20085162.mp3', 'common_voice_it_20095015.mp3', 'common_voice_it_20095016.mp3', 'common_voice_it_20095017.mp3', 'common_voice_it_20095018.mp3', 'common_voice_it_20095019.mp3', 'common_voice_it_20275663.mp3', 'common_voice_it_20275664.mp3', 'common

In [ ]:
# List MP3 files in the Italian 'audio' directory
mp3_files_italian = [f for f in os.listdir(italian_audio_dir) if f.lower().endswith(".mp3")]

if mp3_files_italian:
    print(f"Found {len(mp3_files_italian)} MP3 files in {italian_audio_dir}:")
    print(mp3_files_italian[:10])  # Display first 10 files for brevity
else:
    print(f"No MP3 files found in {italian_audio_dir}. Please upload your MP3 files to proceed.")


Found 62000 MP3 files in /content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/audio:
['common_voice_it_20283956.mp3', 'common_voice_it_20283957.mp3', 'common_voice_it_20283958.mp3', 'common_voice_it_20283959.mp3', 'common_voice_it_20283960.mp3', 'common_voice_it_25331358.mp3', 'common_voice_it_25331359.mp3', 'common_voice_it_25331360.mp3', 'common_voice_it_25331361.mp3', 'common_voice_it_25331362.mp3']


In [ ]:
pip install tqdm librosa soundfile


In [ ]:
# Step 2: Preprocessing - Optimized Conversion of MP3 to WAV for Multiple Languages


import os
import librosa
import soundfile as sf
from tqdm import tqdm
import logging
import numpy as np

# Configurar logging para registrar errores
logging.basicConfig(
    filename='conversion_errors.log',
    level=logging.ERROR,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

def preprocess_audio_file(audio_file, audio_dir, output_dir, sample_rate=16000, target_length=5):
    """
    Procesa y convierte un archivo de audio a WAV con resampling y padding/truncamiento.

    Parámetros:
    - audio_file (str): Nombre del archivo de audio.
    - audio_dir (str): Directorio donde se encuentra el archivo de audio.
    - output_dir (str): Directorio donde se guardará el archivo WAV.
    - sample_rate (int): Tasa de muestreo objetivo.
    - target_length (int): Duración objetivo en segundos.

    Retorna:
    - tuple: (Éxito (bool), Nombre del archivo)
    """
    try:
        audio_path = os.path.join(audio_dir, audio_file)

        # Cargar el audio con librosa, resampleando a sample_rate si es necesario
        waveform, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

        # Calcular el número de muestras objetivo
        target_samples = sample_rate * target_length

        # Padding o truncamiento para hacer que todos los audios tengan la misma duración
        if len(waveform) < target_samples:
            padding = target_samples - len(waveform)
            waveform = np.pad(waveform, (0, padding), 'constant')
        else:
            waveform = waveform[:target_samples]

        # Guardar el archivo en formato WAV solo si no existe
        target_file = os.path.join(output_dir, f"{os.path.splitext(audio_file)[0]}.wav")
        if not os.path.exists(target_file):
            sf.write(target_file, waveform, sample_rate)

        return (True, audio_file)
    except Exception as e:
        logging.error(f"Error procesando {audio_file}: {e}")
        return (False, audio_file)

def preprocess_audio(audio_dir, output_dir, sample_rate=16000, target_length=5):
    """
    Preprocesa y convierte audios de MP3 a WAV de manera secuencial.

    Parámetros:
    - audio_dir (str): Directorio de entrada con archivos de audio.
    - output_dir (str): Directorio de salida para archivos WAV.
    - sample_rate (int): Tasa de muestreo objetivo.
    - target_length (int): Duración objetivo en segundos.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Filtrar solo archivos de audio con extensiones específicas
    valid_extensions = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac')
    audio_files = [f for f in os.listdir(audio_dir) if f.lower().endswith(valid_extensions)]
    total_files = len(audio_files)

    if total_files == 0:
        print(f"No se encontraron archivos de audio en {audio_dir}.")
        return

    print(f"Total de archivos a procesar: {total_files}")

    # Iterar sobre cada archivo con una barra de progreso
    for audio_file in tqdm(audio_files, desc="Procesando audios", unit="archivo"):
        success, file_name = preprocess_audio_file(audio_file, audio_dir, output_dir, sample_rate, target_length)
        if not success:
            print(f"Error procesando {file_name}. Revisa 'conversion_errors.log' para más detalles.")

    # Mostrar el porcentaje final cuando el procesamiento se haya completado
    print(f"Progreso: 100.00% - Todos los archivos procesados.")

# Definir las rutas para cada idioma
languages_conversion = {
    "italian": {
        "audio_dir": r'D:\TinyML project\SST\Data\raw\italian\audio',
        "wav_dir": r'D:\TinyML project\SST\Data\raw\italian\wavpreprocessed'
    },
    "spanish": {
        "audio_dir": r'D:\TinyML project\SST\Data\raw\spanish\audio',
        "wav_dir": r'D:\TinyML project\SST\Data\raw\spanish\wavpreprocessed'
    },
    "swahili": {
        "audio_dir": r'D:\TinyML project\SST\Data\raw\swahili\audio',
        "wav_dir": r'D:\TinyML project\SST\Data\raw\swahili\wavpreprocessed'
    }
}

# Parámetros de configuración
sample_rate = 16000      # Tasa de muestreo objetivo
target_length = 5        # Duración objetivo en segundos

# Iterar sobre cada idioma y realizar la conversión
for lang, dirs in languages_conversion.items():
    print(f"Iniciando procesamiento para el idioma: {lang.capitalize()}")
    preprocess_audio(
        audio_dir=dirs['audio_dir'],
        output_dir=dirs['wav_dir'],
        sample_rate=sample_rate,
        target_length=target_length
    )
    print(f"Finalizado procesamiento para el idioma: {lang.capitalize()}\n")


Number of CPU cores available: 8
Number of parallel worker processes: 7


Copying MP3 files for Italian from Drive to local storage...


KeyboardInterrupt: 

### Clean Text and Generate Phonetics
Preprocess the transcriptions by cleaning the text and generating phonetic transcriptions using Epitran.

In [ ]:
# Step 3: Preprocessing - Cleaning Text, Phonetic Transcription, Creating JSON Files per Language, and Handling Durations

import os
import json
import epitran
import pandas as pd
import re
import string
from tqdm.auto import tqdm  # Usar tqdm.auto para evitar dependencia de ipywidgets
import logging

# Configurar logging para registrar errores
logging.basicConfig(
    filename='conversion_errors.log',
    level=logging.ERROR,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

# Mapping of languages to Epitran codes
language_map = {
    "it": "ita-Latn",    # Italian
    "es": "spa-Latn",    # Spanish
    "sw": "swh-Latn"     # Swahili
    # Agrega más códigos de idioma según sea necesario
}

def preprocess_text(text):
    """
    Limpia el texto de entrada convirtiéndolo a minúsculas, estandarizando la puntuación,
    eliminando caracteres especiales y eliminando espacios extra.

    Parámetros:
    - text (str): El texto a preprocesar.

    Retorna:
    - str: El texto preprocesado.
    """
    if not isinstance(text, str):
        return ""

    # Convertir a minúsculas
    text = text.lower()

    # Eliminar entradas vacías
    if not text.strip():
        return ""

    # Estandarizar la puntuación (mantener instancias únicas)
    punctuations = string.punctuation
    text = re.sub(r'\s+', ' ', text)  # Reemplazar múltiples espacios por uno solo
    text = re.sub(rf"[{re.escape(punctuations)}]+", lambda x: x.group(0)[0], text)

    # Eliminar caracteres especiales (mantener letras, dígitos y puntuación básica)
    allowed_characters = string.ascii_letters + string.digits + " .,!?-"
    text = ''.join(char for char in text if char in allowed_characters)

    # Eliminar espacios extra
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def initialize_epitran(locale):
    """
    Inicializa una instancia de Epitran para el locale dado.

    Parámetros:
    - locale (str): El código del locale (e.g., 'it' para italiano).

    Retorna:
    - Epitran instance o None si falla la inicialización.
    """
    code = language_map.get(locale.lower())
    if code:
        try:
            return epitran.Epitran(code)
        except Exception as e:
            logging.error(f"Error initializing Epitran for locale '{locale}': {e}")
            return None
    else:
        logging.error(f"Locale '{locale}' is not supported by Epitran.")
        return None

def generate_phonetics(text, epi_instance):
    """
    Genera la transcripción fonética del texto usando Epitran.

    Parámetros:
    - text (str): El texto a transcribir.
    - epi_instance (Epitran): La instancia de Epitran inicializada.

    Retorna:
    - str: La transcripción fonética o el texto original si falla la transcripción.
    """
    if epi_instance:
        try:
            return epi_instance.transliterate(text)
        except Exception as e:
            logging.error(f"Error in phonetic transcription: {e}")
            return text  # Fallback al texto original
    else:
        return text  # Fallback al texto original

def process_language(lang, transcription_files_lang, audio_dir, durations_file, json_output):
    """
    Procesa las transcripciones para un idioma específico: limpia el texto, genera fonética,
    asocia con duraciones, y guarda los datos en formato JSON.

    Parámetros:
    - lang (str): El idioma a procesar.
    - transcription_files_lang (dict): Diccionario con rutas a 'validated', 'invalidated' y 'other' TSV.
    - audio_dir (str): Ruta al directorio que contiene archivos WAV.
    - durations_file (str): Ruta al archivo 'clip_durations.tsv'.
    - json_output (str): Ruta donde se guardará el JSON de salida.
    """
    print(f"Processing language: {lang.capitalize()}")

    # Inicializar una lista para almacenar los DataFrames
    df_list = []

    # Leer los archivos TSV y agregar una columna 'status'
    for status, file_path in transcription_files_lang.items():
        if not os.path.exists(file_path):
            logging.error(f"File {status} TSV not found for language '{lang}': {file_path}")
            print(f"File {status} TSV not found for language '{lang}'.")
            continue

        try:
            df = pd.read_csv(file_path, sep='\t', dtype=str)
            df['status'] = status.capitalize()  # Añadir columna de estado ('Validated', 'Invalidated', 'Other')
            df_list.append(df)
            print(f"Loaded {status} transcriptions: {len(df)} entries.")
        except Exception as e:
            logging.error(f"Error loading {status} TSV file for {lang}: {e}")
            print(f"Error loading {status} TSV file for {lang}.")

    # Combinar los DataFrames
    if df_list:
        transcriptions = pd.concat(df_list, ignore_index=True)
        print(f"Total transcriptions (validated + invalidated + other) for {lang}: {len(transcriptions)}")
    else:
        print(f"No transcriptions found for {lang}. Skipping.")
        return

    # Asegurar que las columnas requeridas existen
    required_columns = ['path', 'sentence', 'locale']
    for col in required_columns:
        if col not in transcriptions.columns:
            transcriptions[col] = lang[:2]  # Código de locale por defecto

    # Crear una columna 'base_path' sin extensión y sin ruta
    transcriptions['base_path'] = transcriptions['path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0] if pd.notnull(x) else "")

    # Filtrar las columnas necesarias
    transcriptions = transcriptions[['base_path', 'sentence', 'locale', 'status']]

    # Asegurar que la columna 'sentence' sea de tipo string
    transcriptions['sentence'] = transcriptions['sentence'].astype(str)

    # Cargar las duraciones de los clips
    if not os.path.exists(durations_file):
        logging.error(f"Durations file not found for language '{lang}': {durations_file}")
        print(f"Durations file not found for language '{lang}'.")
        return

    try:
        durations = pd.read_csv(durations_file, sep='\t', dtype=str)
        # Renombrar columnas para consistencia
        durations = durations.rename(columns={'clip': 'path', 'duration[ms]': 'duration_ms'})
        # Crear 'base_path' eliminando la extensión
        durations['base_path'] = durations['path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0] if pd.notnull(x) else "")
        durations = durations[['base_path', 'duration_ms']]
        # Convertir duración de ms a segundos
        durations['duration'] = durations['duration_ms'].astype(float) / 1000.0
        durations = durations[['base_path', 'duration']]
        print(f"Loaded clip durations: {len(durations)} entries.")
    except Exception as e:
        logging.error(f"Error loading durations file for {lang}: {e}")
        print(f"Error loading durations file for {lang}.")
        return

    # Inicializar instancia de Epitran
    first_locale = transcriptions['locale'].iloc[0] if not transcriptions.empty else lang[:2]
    epi_instance = initialize_epitran(first_locale)

    # Listar archivos WAV
    if not os.path.exists(audio_dir):
        logging.error(f"Audio directory not found for language '{lang}': {audio_dir}")
        print(f"Audio directory not found for language '{lang}'.")
        return

    wav_files = [f for f in os.listdir(audio_dir) if f.lower().endswith('.wav')]
    total_files = len(wav_files)
    print(f"Total WAV files found: {total_files}")

    # Crear un diccionario de duraciones para acceso rápido
    durations_dict = pd.Series(durations.duration.values, index=durations.base_path).to_dict()

    processed_data = []
    processed_count = 0
    no_transcription_count = 0

    for wav_file in tqdm(wav_files, desc=f"Processing {lang.capitalize()} Audio", unit="file"):
        wav_file_no_ext = os.path.splitext(wav_file)[0]

        # Buscar transcripción correspondiente usando 'base_path'
        matching_rows = transcriptions[transcriptions['base_path'] == wav_file_no_ext]

        if not matching_rows.empty:
            row = matching_rows.iloc[0]
            text = row['sentence']
            locale = row['locale']
            status = row['status']

            # Obtener duración del audio
            duration = durations_dict.get(wav_file_no_ext, None)
            if duration is None:
                logging.error(f"Duration not found for audio file: {wav_file}")
                continue  # Opcional: decidir si omitir o continuar

            # Preprocesar texto
            preprocessed_text = preprocess_text(text)

            # Generar transcripción fonética
            phonetics = generate_phonetics(preprocessed_text, epi_instance)

            # Definir etiquetas de idioma
            language_tag = locale.lower()  # Según el ejemplo, solo el código del idioma

            # Crear entrada para JSON con los campos deseados
            entry = {
                "audio_file": wav_file,
                "text": preprocessed_text,
                "phonetics": phonetics,
                "language_tag": language_tag
            }

            # Añadir a los datos procesados
            processed_data.append(entry)
            processed_count += 1
        else:
            no_transcription_count += 1

    # Calcular estadísticas
    found_percentage = (processed_count / total_files) * 100 if total_files > 0 else 0
    not_found_count = no_transcription_count
    not_found_percentage = (not_found_count / total_files) * 100 if total_files > 0 else 0

    # Guardar datos procesados en JSON como un array
    try:
        with open(json_output, 'w', encoding='utf-8') as f:
            json.dump(processed_data, f, ensure_ascii=False, indent=4)
        print(f"Processed data saved to: {json_output}")
        print(f"Total transcriptions processed for {lang.capitalize()}: {processed_count} ({found_percentage:.2f}%)")
        print(f"Total audio files without transcription: {not_found_count} ({not_found_percentage:.2f}%)\n")
    except Exception as e:
        logging.error(f"Error saving JSON for {lang}: {e}")
        print(f"Error saving JSON for {lang}: {e}\n")

# Definir rutas para cada idioma (validated.tsv, invalidated.tsv y other.tsv)
transcription_files = {
    "italian": {
        "validated": r'D:\TinyML project\SST\Data\raw\italian\validated.tsv',
        "invalidated": r'D:\TinyML project\SST\Data\raw\italian\invalidated.tsv',
        "other": r'D:\TinyML project\SST\Data\raw\italian\other.tsv'
    },
    "spanish": {
        "validated": r'D:\TinyML project\SST\Data\raw\spanish\validated.tsv',
        "invalidated": r'D:\TinyML project\SST\Data\raw\spanish\invalidated.tsv',
        "other": r'D:\TinyML project\SST\Data\raw\spanish\other.tsv'
    },
    "swahili": {
        "validated": r'D:\TinyML project\SST\Data\raw\swahili\validated.tsv',
        "invalidated": r'D:\TinyML project\SST\Data\raw\swahili\invalidated.tsv',
        "other": r'D:\TinyML project\SST\Data\raw\swahili\other.tsv'
    }
}

audio_dirs_json = {
    "italian": r'D:\TinyML project\SST\Data\raw\italian\wavpreprocessed',
    "spanish": r'D:\TinyML project\SST\Data\raw\spanish\wavpreprocessed',
    "swahili": r'D:\TinyML project\SST\Data\raw\swahili\wavpreprocessed'
}

durations_files = {
    "italian": r'D:\TinyML project\SST\Data\raw\italian\clip_durations.tsv',
    "spanish": r'D:\TinyML project\SST\Data\raw\spanish\clip_durations.tsv',
    "swahili": r'D:\TinyML project\SST\Data\raw\swahili\clip_durations.tsv'
}

json_files_output = {
    "italian": r'D:\TinyML project\SST\Data\raw\italian\transcriptionsit.json',
    "spanish": r'D:\TinyML project\SST\Data\raw\spanish\transcriptionses.json',
    "swahili": r'D:\TinyML project\SST\Data\raw\swahili\transcriptionssw.json'
}

# Crear directorios de salida si no existen y limpiar archivos JSON
for lang in json_files_output.keys():
    json_dir = os.path.dirname(json_files_output[lang])
    if not os.path.exists(json_dir):
        os.makedirs(json_dir)
    # Limpiar archivo JSON antes de empezar
    open(json_files_output[lang], 'w').close()

# Procesar cada idioma secuencialmente
for lang in transcription_files.keys():
    process_language(
        lang=lang,
        transcription_files_lang=transcription_files[lang],
        audio_dir=audio_dirs_json[lang],
        durations_file=durations_files[lang],
        json_output=json_files_output[lang]
    )


In [ ]:
#En caso de que se necesite asignar fonetica

import json
import logging
import pandas as pd
import os

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Definir la ruta al archivo de transcripciones y al directorio de audios
transcription_file = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/validatedsw.tsv"
audio_dir = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/wav"

# Cargar transcripciones
transcriptions = pd.read_csv(transcription_file, sep='\t', dtype={'sentence': str})

# Convertir las transcripciones en un diccionario con el nombre del archivo (sin extensión) como clave
transcription_dict = dict(zip(transcriptions['path'].str.replace('.mp3', ''), transcriptions['sentence']))

# Función para generar la transcripción fonética (aproximación basada en reglas simples)
def generate_phonetics_swahili(text):
    phonetic_map = {
        'a': 'a', 'b': 'b', 'c': 'tʃ', 'd': 'd', 'e': 'e', 'f': 'f', 'g': 'ɡ', 'h': 'h', 'i': 'i',
        'j': 'dʒ', 'k': 'k', 'l': 'l', 'm': 'm', 'n': 'n', 'o': 'o', 'p': 'p', 'r': 'r', 's': 's',
        't': 't', 'u': 'u', 'v': 'v', 'w': 'w', 'y': 'j', 'z': 'z', 'ch': 'tʃ', 'sh': 'ʃ', 'ng': 'ŋ'
    }
    phonetics = ''.join(phonetic_map.get(char, char) for char in text)
    return phonetics

# Crear la lista de datos para el archivo JSON
data = []
audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]
total_files = len(audio_files)
logging.info(f"Total de archivos de audio a procesar: {total_files}")

for idx, audio_file in enumerate(audio_files):
    audio_file_no_ext = os.path.splitext(audio_file)[0]

    # Buscar la transcripción correspondiente
    if audio_file_no_ext in transcription_dict:
        text = transcription_dict[audio_file_no_ext]
        phonetics = generate_phonetics_swahili(text)
        data.append({
            "audio_file": audio_file,
            "text": text,
            "phonetics": phonetics
        })

        # Mostrar progreso cada 100 archivos procesados
        if (idx + 1) % 100 == 0 or (idx + 1) == total_files:
            logging.info(f"Progreso: {idx + 1}/{total_files} archivos de audio procesados.")

# Guardar los datos en un archivo JSON
json_file = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/transcriptionsessw.json"
with open(json_file, 'w') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

logging.info(f"Transcripciones fonéticas guardadas en {json_file}")


Paso 2: Cargar y preprocesar los datos

ejemplo de CSV para cargar los archivos

wav_filename,wav_filesize,transcript

audio1.wav,123456,Hola, ¿cómo estás?,es

audio2.wav,234567,Estoy bien, gracias.,es

audio3.wav,345678,Habari, sw

In [ ]:
[
    {
        "audio_file": "audio1.wav",
        "text": "Example sentence in Swahili",
        "phonetics": "eːksɑːmpleː seːnteːnkeː iːn svɑːhiːliː"
        "language_tag": "sw"
    },
    {
        "audio_file": "audio2.wav",
        "text": "Another example sentence",
        "phonetics": "əˈnʌðər ɪɡˈzæmpl ˈsɛntəns"
        "language_tag": "sw"
    }
]

### Optimize Audio
Enhance the quality of audio files by trimming silence, normalizing volume levels, and reducing background noise.



In [ ]:
pip install librosa soundfile noisereduce tqdm


In [ ]:
!pip install noisereduce


In [ ]:
import sys
!{sys.executable} -m pip install noisereduce


In [ ]:
# Step 4: Preprocessing - Audio Optimization (Silence Trimming, Normalization, and Noise Reduction)

import os
import librosa
import soundfile as sf
import noisereduce as nr
import numpy as np
from tqdm.auto import tqdm  # Compatibilidad con Jupyter
import logging

# Configurar logging para registrar solo errores
logging.basicConfig(
    filename='conversion_errors.log',
    level=logging.ERROR,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

# Configuración para cada idioma con rutas de Windows
languages_audio = {
    "italian": r'D:\TinyML project\SST\Data\raw\italian\wavpreprocessed',
    "spanish": r'D:\TinyML project\SST\Data\raw\spanish\wavpreprocessed',
    "swahili": r'D:\TinyML project\SST\Data\raw\swahili\wavpreprocessed'
}

# Parámetros
TARGET_SR = 16000  # Frecuencia de muestreo objetivo
NOISE_DURATION = 0.5  # Duración en segundos para la muestra de ruido
TOP_DB = 10  # Umbral menos agresivo para recorte de silencios
MAX_TRIM_RATIO = 0.2  # Máximo ratio de recorte permitido

def conditional_trim(y, sr, top_db=10, max_trim_ratio=0.2):
    """
    Recorta silencios solo si la proporción de recorte no excede max_trim_ratio.

    Parámetros:
    - y (np.ndarray): Señal de audio.
    - sr (int): Frecuencia de muestreo.
    - top_db (int): Umbral para considerar silencio.
    - max_trim_ratio (float): Proporción máxima de recorte permitida.

    Retorna:
    - np.ndarray: Señal de audio recortada o original.
    """
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    trim_ratio = (len(y) - len(y_trimmed)) / len(y)
    if trim_ratio > max_trim_ratio:
        return y  # No recortar si excede el ratio
    return y_trimmed

def preprocess_audio(input_path, output_path, target_sr=16000, noise_duration=0.5, top_db=10, max_trim_ratio=0.2):
    """
    Preprocesa un archivo de audio recortando silencios condicionalmente, normalizando el volumen y reduciendo el ruido de fondo.

    Parámetros:
    - input_path (str): Ruta al archivo WAV de entrada.
    - output_path (str): Ruta donde se guardará el archivo WAV procesado.
    - target_sr (int): Frecuencia de muestreo objetivo.
    - noise_duration (float): Duración en segundos para capturar la muestra de ruido.
    - top_db (int): Umbral para recorte de silencios.
    - max_trim_ratio (float): Proporción máxima de recorte permitida.
    """
    try:
        # Cargar audio con librosa
        y, sr = librosa.load(input_path, sr=target_sr, mono=True)

        # Recortar silencios condicionalmente
        y_trimmed = conditional_trim(y, sr, top_db, max_trim_ratio)

        # Normalizar audio al pico de 0.99
        if np.max(np.abs(y_trimmed)) > 0:
            y_normalized = y_trimmed / np.max(np.abs(y_trimmed)) * 0.99
        else:
            y_normalized = y_trimmed

        # Extraer muestra de ruido desde el principio (primeros NOISE_DURATION segundos)
        noise_sample = y_normalized[:int(noise_duration * target_sr)]

        # Reducir ruido
        y_denoised = nr.reduce_noise(y=y_normalized, sr=target_sr, y_noise=noise_sample, prop_decrease=1.0)

        # Guardar el audio procesado
        sf.write(output_path, y_denoised, target_sr)

    except Exception as e:
        logging.error(f"Error processing {input_path}: {e}")

def optimize_audio(language, audio_dir, batch_size=1000):
    """
    Optimiza todos los archivos WAV en el directorio especificado recortando silencios condicionalmente, normalizando y reduciendo el ruido.
    Procesa los archivos en lotes para manejar mejor los recursos.

    Parámetros:
    - language (str): El idioma que se está procesando (e.g., 'italian').
    - audio_dir (str): Ruta al directorio que contiene los archivos WAV.
    - batch_size (int): Número de archivos a procesar en cada lote.
    """
    # Definir directorio de salida para audios optimizados
    optimized_dir = os.path.join(audio_dir, "optimized")
    os.makedirs(optimized_dir, exist_ok=True)
    print(f"Optimizing audio for language: {language.capitalize()}")
    print(f"Output directory: {optimized_dir}")

    # Listar todos los archivos WAV
    wav_files = [f for f in os.listdir(audio_dir) if f.lower().endswith('.wav')]
    total_files = len(wav_files)
    print(f"Found {total_files} WAV files to optimize.\n")

    if total_files == 0:
        print(f"No se encontraron archivos WAV en {audio_dir} para {language.capitalize()}.\n")
        return

    # Procesar en lotes
    for i in range(0, total_files, batch_size):
        batch = wav_files[i:i + batch_size]
        batch_number = i // batch_size + 1
        total_batches = (total_files - 1) // batch_size + 1
        print(f"Processing batch {batch_number} of {total_batches} for {language.capitalize()}")
        for wav_file in tqdm(batch, desc=f"{language.capitalize()} - Batch {batch_number}", unit="file"):
            input_path = os.path.join(audio_dir, wav_file)
            output_path = os.path.join(optimized_dir, wav_file)
            preprocess_audio(input_path, output_path, TARGET_SR, NOISE_DURATION, TOP_DB, MAX_TRIM_RATIO)
        print(f"Completed batch {batch_number} of {total_batches} for {language.capitalize()}\n")

    print(f"Completed audio optimization for: {language.capitalize()}\n")

# Optimizar audio para cada idioma
for lang, audio_dir in languages_audio.items():
    optimize_audio(lang, audio_dir)


In [ ]:
#multiprocessign TO ASSESS:

import multiprocessing

def process_file(wav_file, audio_dir, processed_dir, target_sr, noise_duration):
    input_path = os.path.join(audio_dir, wav_file)
    output_path = os.path.join(processed_dir, wav_file)
    preprocess_audio(input_path, output_path, target_sr, noise_duration)

for lang, audio_dir in languages.items():
    print(f"Procesando idioma: {lang.capitalize()}")

    # Definir directorio para audios preprocesados
    processed_dir = os.path.join(audio_dir, "wavpreprocessed")
    os.makedirs(processed_dir, exist_ok=True)

    # Listar archivos WAV en el directorio actual
    wav_files = [f for f in os.listdir(audio_dir) if f.lower().endswith('.wav')]
    total_files = len(wav_files)

    # Configurar el pool de procesos
    pool = multiprocessing.Pool(processes=4)  # Ajusta el número de procesos según tus recursos

    # Usar tqdm para mostrar progreso
    for _ in tqdm(pool.imap_unordered(lambda f: process_file(f, audio_dir, processed_dir, TARGET_SR, NOISE_DURATION), wav_files), total=total_files, desc=f"Procesando {lang.capitalize()}", unit="archivo"):
        pass

    pool.close()
    pool.join()

    print(f"Finalizado procesamiento para: {lang.capitalize()}\n")


### Verify Text-Audio Alignment
Ensure that each transcription corresponds to its respective audio file.

In [ ]:
# Step 5: Preprocessing - Alignment of Text and Audio

import os
import json
from tqdm.auto import tqdm  # Usar tqdm.auto para compatibilidad en Jupyter
import logging

# Configurar logging para registrar errores (usa el mismo archivo que Steps 3 y 4)
logging.basicConfig(
    filename='conversion_errors.log',
    level=logging.ERROR,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

def verify_alignment(json_file, audio_dir, lang):
    """
    Verifica que cada entrada en el archivo JSON tenga un archivo de audio correspondiente en el directorio especificado.

    Parámetros:
    - json_file (str): Ruta al archivo JSON que contiene las transcripciones.
    - audio_dir (str): Ruta al directorio que contiene los archivos WAV optimizados.
    - lang (str): Nombre del idioma para propósitos de logging.
    """
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)  # Cargar todo el JSON como una lista
    except Exception as e:
        logging.error(f"Error al cargar el archivo JSON para {lang}: {e}")
        print(f"Error al cargar el archivo JSON para {lang}. Revisa 'conversion_errors.log' para más detalles.")
        return

    # Obtener el conjunto de archivos de audio disponibles
    try:
        audio_files = set([f.lower() for f in os.listdir(audio_dir) if f.lower().endswith('.wav')])
    except Exception as e:
        logging.error(f"Error al acceder al directorio de audio para {lang}: {e}")
        print(f"Error al acceder al directorio de audio para {lang}. Revisa 'conversion_errors.log' para más detalles.")
        return

    # Lista para almacenar los archivos de audio faltantes
    missing_audios = []

    # Verificar cada entrada en el JSON
    for entry in tqdm(data, desc=f"Verificando alineación para {lang.capitalize()}", unit="entrada"):
        audio_filename = entry.get('audio_file', '').strip().lower()
        if not audio_filename:
            logging.error(f"Entrada sin 'audio_file' en JSON para {lang}: {entry}")
            missing_audios.append("Unnamed audio file")
            continue
        # Asegurarse de que el archivo tiene extensión .wav
        if not audio_filename.endswith('.wav'):
            audio_filename = os.path.splitext(audio_filename)[0] + '.wav'
        if audio_filename not in audio_files:
            missing_audios.append(audio_filename)

    # Reportar resultados
    if missing_audios:
        print(f"\nIdioma: {lang.capitalize()}")
        print(f"Se encontraron {len(missing_audios)} archivos de audio faltantes en {audio_dir}:")
        for audio in missing_audios[:10]:  # Mostrar solo los primeros 10
            print(f"- {audio}")
        if len(missing_audios) > 10:
            print(f"... y {len(missing_audios) - 10} más.")
        print("\n")
    else:
        print(f"\nTodos los audios están correctamente alineados en {audio_dir} para {lang.capitalize()}.\n")

# Definir JSON files y directorios de audio optimizados con rutas de Windows
json_files_alignment = {
    "italian": r'D:\TinyML project\SST\Data\raw\italian\transcriptionsit.json',
    "spanish": r'D:\TinyML project\SST\Data\raw\spanish\transcriptionses.json',
    "swahili": r'D:\TinyML project\SST\Data\raw\swahili\transcriptionssw.json'
}

audio_dirs_alignment = {
    "italian": r'D:\TinyML project\SST\Data\raw\italian\wavpreprocessed\optimized',
    "spanish": r'D:\TinyML project\SST\Data\raw\spanish\wavpreprocessed\optimized',
    "swahili": r'D:\TinyML project\SST\Data\raw\swahili\wavpreprocessed\optimized'
}

# Verificar alineación para cada idioma
for lang in json_files_alignment.keys():
    print(f"Iniciando verificación de alineación para el idioma: {lang.capitalize()}")
    verify_alignment(json_files_alignment[lang], audio_dirs_alignment[lang], lang)
    print(f"Finalizada verificación de alineación para el idioma: {lang.capitalize()}\n")


In [ ]:
pip install sentencepiece tqdm


### Tokenization and Dataset Creation
Tokenize the transcriptions, pad the sequences, encode language tags, and create a TensorFlow dataset ready for training.

In [ ]:
# Step 6: Preprocessing - Tokenization, Padding, Encoding Language Tags, and Creating TensorFlow Dataset

import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import matplotlib.pyplot as plt

def main():
    # ==========================
    # 1. Configuración de Rutas
    # ==========================

    original_json_files = {
        "it": r'D:\TinyML project\SST\Data\raw\italian\transcriptionsit.json',
        "es": r'D:\TinyML project\SST\Data\raw\spanish\transcriptionses.json',
        "sw": r'D:\TinyML project\SST\Data\raw\swahili\transcriptionssw.json'
    }

    # ==========================
    # 2. Preparar los Datos para Crear el Tokenizer
    # ==========================

    # Cargar todos los textos y fonéticas para crear el tokenizer
    all_texts = []
    all_phonetics = []

    for lang_code, json_path in original_json_files.items():
        if not os.path.exists(json_path):
            print(f"Archivo no encontrado: {json_path}. Se omitirá este idioma.\n")
            continue

        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print(f"Error al cargar el archivo JSON para {lang_code}: {e}\n")
            continue

        for entry in data:
            text = entry.get('text', '').strip()
            phonetic = entry.get('phonetics', '').strip()
            language_tag = entry.get('language_tag', '').strip().lower()
            if text and phonetic and language_tag:
                # Prependemos el language_tag al texto y fonética
                text_with_lang = f"{language_tag} {text}"
                phonetic_with_lang = f"{language_tag} {phonetic}"
                all_texts.append(text_with_lang)
                all_phonetics.append(phonetic_with_lang)

    if not all_texts or not all_phonetics:
        print("No se han cargado datos para crear el tokenizer.\n")
        return
    else:
        print("Datos cargados correctamente para crear el tokenizer.\n")

    # ==========================
    # 3. Crear el Tokenizer de Keras Personalizado
    # ==========================

    # Combinar textos y fonéticas para ajustar el tokenizer
    tokenizer_texts = all_texts + all_phonetics

    # Definir filtros personalizados (no eliminar ningún carácter)
    custom_filters = ''  # No filtramos ningún carácter

    # Crear el tokenizer
    tokenizer = Tokenizer(
        filters=custom_filters,
        lower=False,
        split=' ',
        char_level=False,
        oov_token='<unk>'
    )

    # Ajustar el tokenizer a tus datos
    tokenizer.fit_on_texts(tokenizer_texts)

    # Guardar el tokenizer para uso futuro
    tokenizer_save_path = r'D:\TinyML project\SST\Data\preprocessed_encoders\keras_tokenizer.pkl'
    os.makedirs(os.path.dirname(tokenizer_save_path), exist_ok=True)

    with open(tokenizer_save_path, 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

    print("Tokenizer de Keras guardado en:", tokenizer_save_path)

    # ==========================
    # 4. Verificar Inclusión de Símbolos Especiales
    # ==========================

    special_symbols = ['ɾ', 'ɓ', 'ɠ', '͡']
    word_index = tokenizer.word_index

    for symbol in special_symbols:
        if symbol in word_index:
            print(f"Símbolo '{symbol}' está en el tokenizer con índice {word_index[symbol]}")
        else:
            print(f"Símbolo '{symbol}' NO está en el tokenizer")

    # ==========================
    # 5. Procesar y Guardar los Datos Tokenizados por Idioma
    # ==========================

    for lang_code, json_path in original_json_files.items():
        print(f"\nProcesando idioma: {lang_code}")

        if not os.path.exists(json_path):
            print(f"Archivo no encontrado: {json_path}. Se omitirá este idioma.")
            continue

        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            print(f"Cargadas {len(data)} entradas desde {json_path}")
        except Exception as e:
            print(f"Error al cargar el archivo JSON para {lang_code}: {e}")
            continue

        texts = []
        phonetics = []
        language_tags = []
        audio_files = []

        for entry in data:
            text = entry.get('text', '').strip()
            phonetic = entry.get('phonetics', '').strip()
            language_tag = entry.get('language_tag', '').strip().lower()
            audio_file = entry.get('audio_file', '').strip()

            if text and phonetic and language_tag:
                # Prependemos el language_tag al texto y fonética
                text_with_lang = f"{language_tag} {text}"
                phonetic_with_lang = f"{language_tag} {phonetic}"

                texts.append(text_with_lang)
                phonetics.append(phonetic_with_lang)
                language_tags.append(language_tag)
                audio_files.append(audio_file)
            else:
                print(f"Entrada omitida por campos incompletos en {json_path}")

        if not texts or not phonetics:
            print(f"No se han cargado datos para el idioma {lang_code}.")
            continue
        else:
            print(f"Datos cargados correctamente para el idioma {lang_code}.")

        # Tokenizar las secuencias modificadas
        text_sequences = tokenizer.texts_to_sequences(texts)
        phonetics_sequences = tokenizer.texts_to_sequences(phonetics)

        # Calcular la longitud máxima de secuencia para este idioma (opcional)
        lengths_text = [len(seq) for seq in text_sequences]
        lengths_phonetics = [len(seq) for seq in phonetics_sequences]

        max_length_text = int(np.percentile(lengths_text, 95))
        max_length_phonetics = int(np.percentile(lengths_phonetics, 95))
        max_length = max(max_length_text, max_length_phonetics)

        print(f"Estableciendo max_length a {max_length} para el idioma {lang_code} basado en el percentil 95 de las longitudes.\n")

        # Aplicar padding a las secuencias
        padded_text_sequences = pad_sequences(
            text_sequences,
            maxlen=max_length,
            padding='post',
            truncating='post',
            value=0
        )

        padded_phonetics_sequences = pad_sequences(
            phonetics_sequences,
            maxlen=max_length,
            padding='post',
            truncating='post',
            value=0
        )

        print(f"Secuencias de tokens padding a una longitud de {max_length} tokens para el idioma {lang_code}.\n")

        # Crear una lista de diccionarios con los datos tokenizados
        tokenized_data = []
        for idx in range(len(audio_files)):
            entry = {
                "audio_file": audio_files[idx],
                "text": texts[idx],
                "phonetics": phonetics[idx],
                "language_tag": language_tags[idx],
                "token_sequence_text": text_sequences[idx],
                "token_sequence_phonetics": phonetics_sequences[idx]
            }
            tokenized_data.append(entry)

        # Guardar los datos tokenizados para este idioma
        lang_dir = os.path.dirname(json_path)
        base_name = os.path.splitext(os.path.basename(json_path))[0]
        tokenized_json_path = os.path.join(lang_dir, f"{base_name}Tok.json")

        try:
            with open(tokenized_json_path, 'w', encoding='utf-8') as f:
                json.dump(tokenized_data, f, ensure_ascii=False, indent=4)
            print(f"Datos tokenizados guardados en: {tokenized_json_path}")
        except Exception as e:
            print(f"Error al guardar los datos tokenizados para {lang_code}: {e}")

        # Guardar los tensores para este idioma
        tensors = {
            "audio_file": np.array(audio_files),
            "input_ids_text": padded_text_sequences,
            "input_ids_phonetics": padded_phonetics_sequences,
            "language_id": np.array(language_tags)
        }

        tensors_save_dir = os.path.join(lang_dir, 'tokenized_tensors')
        os.makedirs(tensors_save_dir, exist_ok=True)
        tensors_save_path = os.path.join(tensors_save_dir, f"{base_name}_tensors.npz")

        try:
            np.savez_compressed(tensors_save_path, **tensors)
            print(f"Tensores guardados en: {tensors_save_path}\n")
        except Exception as e:
            print(f"Error al guardar los tensores para {lang_code}: {e}")

    # ==========================
    # 6. Guardar el Label Encoder para Uso Futuro
    # ==========================

    label_encoder_save_dir = r'D:\TinyML project\SST\Data\preprocessed_encoders'
    os.makedirs(label_encoder_save_dir, exist_ok=True)
    label_encoder_path = os.path.join(label_encoder_save_dir, 'language_label_encoder.pkl')

    # Recolectar todas las etiquetas de lenguaje de los datos procesados
    all_language_tags = list(original_json_files.keys())

    # Codificar las etiquetas de lenguaje
    le = LabelEncoder()
    le.fit(all_language_tags)
    print(f"Clases de lenguaje encontradas: {le.classes_}\n")

    try:
        with open(label_encoder_path, 'wb') as le_file:
            pickle.dump(le, le_file)
        print(f"Label Encoder guardado en: {label_encoder_path}")
    except Exception as e:
        print(f"Error al guardar el Label Encoder: {e}")

if __name__ == '__main__':
    main()



In [ ]:
# Check the tokenization if it's correct or not

import pickle

def verificar_tokenizacion(token_ids):
    # Ruta al archivo del Tokenizer guardado
    tokenizer_path = r'D:\TinyML project\SST\Data\preprocessed_encoders\keras_tokenizer.pkl'

    # Cargar el Tokenizer de Keras
    with open(tokenizer_path, 'rb') as handle:
        tokenizer = pickle.load(handle)
    print("Tokenizer cargado exitosamente.")

    # Crear un diccionario inverso para mapear IDs a palabras
    reverse_word_index = dict([(value, key) for (key, value) in tokenizer.word_index.items()])

    # Decodificar los IDs de tokens a palabras
    decoded_tokens = [reverse_word_index.get(token_id, '<unk>') for token_id in token_ids]

    # Mostrar el resultado
    print("Tokens decodificados:")
    for idx, token in zip(token_ids, decoded_tokens):
        print(f"ID {idx}: '{token}'")

# Ejemplo de uso
if __name__ == "__main__":
    # Lista de IDs de tokens que deseas decodificar
    token_ids = [ 4, 8186, 19572, 91, 43646]

    verificar_tokenizacion(token_ids)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ruta a los archivos de audio y transcripciones en Google Drive
audio_dir = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/wav"
transcription_file = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/transcriptionsit.json"

# **TENSORFLOW MODEL**

In [ ]:
# ===============================================
# Celda 1: Verificación de la GPU
# ===============================================
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🟢 GPU(s) detectada(s): {len(gpus)}")
    for gpu in gpus:
        print(f"  - {gpu.name}")
else:
    print("❌ No se encontraron GPUs. Asegúrate de haber activado la GPU en `Entorno de ejecución` > `Cambiar tipo de entorno de ejecución`.")


🟢 GPU(s) detectada(s): 1
  - /physical_device:GPU:0


In [ ]:
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64/cuda-ubuntu2004.pin
!sudo mv cuda-ubuntu2004.pin /etc/apt/preferences.d/cuda-repository-pin-600
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda-repo-ubuntu2004-11-8-local_11.8.0-520.61.05-1_amd64.deb
!sudo dpkg -i cuda-repo-ubuntu2004-11-8-local_11.8.0-520.61.05-1_amd64.deb
!sudo apt-key add /var/cuda-repo-ubuntu2004-11-8-local/7fa2af80.pub
!sudo apt-get update
!sudo apt-get -y install cuda-toolkit-11-8

--2025-01-11 16:15:09--  https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64/cuda-ubuntu2004.pin
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.45.12.25, 23.45.12.18
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.45.12.25|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 190 [text/plain]
Saving to: ‘cuda-ubuntu2004.pin’

cuda-ubuntu2004.pin 100%[===================>]     190  --.-KB/s    in 0s      

2025-01-11 16:15:09 (53.8 MB/s) - ‘cuda-ubuntu2004.pin’ saved [190/190]

--2025-01-11 16:15:09--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda-repo-ubuntu2004-11-8-local_11.8.0-520.61.05-1_amd64.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.45.12.25, 23.45.12.18
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.45.12.25|:443... connected.
HTTP request sent, awaiting response... 200 O

In [ ]:
!pip uninstall -y keras tensorflow transformers

Found existing installation: keras 3.5.0
Uninstalling keras-3.5.0:
  Successfully uninstalled keras-3.5.0
Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Successfully uninstalled tensorflow-2.17.1
Found existing installation: transformers 4.47.1
Uninstalling transformers-4.47.1:
  Successfully uninstalled transformers-4.47.1


In [ ]:
!pip install tensorflow==2.10.0  # O la versión compatible con tu entorno
!pip install typing-extensions<4.6.0,>=3.6.6  # Según el requerimiento de tensorflow 2.13.0
!pip install tf-keras==2.10.0  # O la versión compatible con tu tensorflow
!pip install librosa --upgrade
!pip install jiwer
!pip install gradio
!pip install torch
!pip install transformers
!pip install pandas
!pip install tqdm
!pip install tensorboard
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.0/578.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.7/438.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.3/781.3 kB 58.6 MB/s eta 0:00:00
  Attempting uninstall: tensorboard-data-server
    Found existing installation: tensorboard-data-server 0.7.2
    Uninstalling tensorboard-data-server-0.7.2:
      Successfully uninstalled tensorboard-data-server-0.7.2
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.5
    Uninstalling protobuf-4.25.5:
      Successfully uninstalled pro

/bin/bash: line 1: 4.6.0,: No such file or directory
ERROR: Could not find a version that satisfies the requirement tf-keras==2.10.0 (from versions: 2.14.1, 2.15.0rc0, 2.15.0rc1, 2.15.0, 2.15.1rc0, 2.15.1, 2.16.0rc0, 2.16.0rc1, 2.16.0rc2, 2.16.0rc3, 2.16.0rc4, 2.16.0, 2.17.0rc0, 2.17.0, 2.18.0rc0, 2.18.0)
ERROR: No matching distribution found for tf-keras==2.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 7.9 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0

In [ ]:
# Celda 2: Importar las librerías y montar Google Drive
import os
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split
from jiwer import wer, cer
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from tqdm import tqdm
import psutil
import gc
import json

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Celda 3: Definir las rutas de acceso a datos y modelos

# Directorios de audio y transcripciones
audio_dirs = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/wavpreprocessed",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/wavpreprocessed",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/wavpreprocessed"
}

tokenized_json_paths = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/transcriptionsitTok.json",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/transcriptionsesTok.json",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/transcriptionsswTok.json"
}

# Directorio base para datos procesados y modelos
base_dir = '/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual'

# Directorio para processed_features
processed_dirs = {
    "Italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/Italian/processed_features",
    "Spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/Spanish/processed_features",
    "Swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/Swahili/processed_features"
}

# Modelo final
model_save_path = os.path.join(base_dir, 'final_model.h5')

# Checkpoints y logs
checkpoint_dir = os.path.join(base_dir, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

logs_dir = os.path.join(base_dir, 'logs')
os.makedirs(logs_dir, exist_ok=True)

# Idiomas y sus codigos
languages = {
    'Italian': 'it',
    'Spanish': 'es',
    'Swahili': 'sw'
}


In [ ]:
# ===============================================
# Celda 3.1: Mantener la Sesión Activa en Google Colab
# ===============================================

from IPython.display import HTML, display

def keep_alive():
    display(HTML('''
        <script>
            function ClickConnect(){
                console.log("Clicking");
                document.querySelector("colab-toolbar-button#connect").click()
            }
            setInterval(ClickConnect, 60000) // Click every 60 seconds
        </script>
    '''))

keep_alive()


In [ ]:
# ================================================
# Celda 4: Alineación de audios (SST Alignment)
# ================================================

import os
import numpy as np
import json
from tqdm import tqdm
import shutil
from concurrent.futures import ThreadPoolExecutor

# Parámetros de alineación
TARGET_SHAPE = (20, 198)  # Dimensión deseada de cada mel-spectrogram
SHOW_WARNINGS = True
MAX_WARNINGS = 10
SAVE_INTERVAL = 1000
NUM_THREADS = 4  # Número de hilos para procesamiento paralelo

# Directorio donde guardaremos los datos alineados
ALIGNED_DATA_DIR = os.path.join(base_dir, "aligned_data")
os.makedirs(ALIGNED_DATA_DIR, exist_ok=True)


# ----------------------------------------------
# Funciones auxiliares
# ----------------------------------------------
def resize_mel(mel, target_shape):
    """
    Recorta/padding para ajustar el mel-spectrogram al tamaño TARGET_SHAPE.
    """
    target_rows, target_cols = target_shape
    rows, cols = mel.shape

    # Recortar si excede
    mel = mel[:target_rows, :target_cols]

    # Padding si faltan filas o columnas
    pad_rows = max(0, target_rows - rows)
    pad_cols = max(0, target_cols - cols)
    if pad_rows > 0 or pad_cols > 0:
        mel = np.pad(mel, ((0, pad_rows), (0, pad_cols)), mode='constant')

    return mel

def backup_file(file_path):
    """
    Hace una copia de seguridad con extensión .bak, si el archivo existe.
    """
    if os.path.exists(file_path):
        backup_path = file_path + ".bak"
        shutil.copy(file_path, backup_path)
        print(f"Copia de seguridad creada: {backup_path}")

def save_progress(language_name,
                  aligned_data_path,
                  aligned_filenames_path,
                  existing_mel,
                  existing_ids,
                  current_mel,
                  current_ids,
                  aligned_filenames):
    """
    Guarda los datos alineados en formato .npz comprimido,
    uniendo los datos existentes con los actuales.
    - mel_spectrograms se apila con np.stack, asumiendo que
      todos están en shape (20,198).
    - input_ids se guarda como dtype=object (listas de long. variable).
    """
    try:
        backup_file(aligned_data_path)  # Copia de seguridad antes de sobreescribir

        # Combinar mel spectrograms (todos deben ser (20,198) tras resize_mel)
        all_mel = existing_mel + current_mel
        mel_array = np.stack(all_mel, axis=0)  # shape: (N, 20, 198)

        # Combinar token sequences (varias longitudes)
        all_ids = existing_ids + current_ids
        ids_array = np.array(all_ids, dtype=object)

        # Guardado comprimido
        np.savez_compressed(
            aligned_data_path,
            mel_spectrograms=mel_array,
            input_ids=ids_array
        )

        # Guardar también la lista de filenames ya alineados
        with open(aligned_filenames_path, 'w') as f:
            for fn in aligned_filenames:
                f.write(fn + '\n')

        print(f"Progreso guardado en {aligned_data_path}: {len(all_mel)} muestras procesadas.")
    except Exception as e:
        print(f"Error al guardar progreso incremental: {e}")

def process_entry(entry, features_dir):
    """
    Función para cada hilo (ThreadPoolExecutor):
    - Retorna (mel, token_sequence, filename) si todo va bien;
      de lo contrario (None, None, filename) para loguear warning.
    """
    filename = entry.get('audio_file', '')
    token_sequence = entry.get('token_sequence_text', [])

    # Chequeos mínimos
    if not filename or not token_sequence:
        return None, None, None

    # Ruta al .npy de features (según la carpeta declarada en 'processed_dirs')
    npy_filename = os.path.splitext(filename)[0] + '.npy'
    feature_path = os.path.join(features_dir, npy_filename)

    # Si no existe la característica
    if not os.path.exists(feature_path):
        return None, None, filename

    # Intentar cargar el mel-spectrogram
    try:
        mel = np.load(feature_path, allow_pickle=True)
        mel = resize_mel(mel, TARGET_SHAPE)
        return mel, token_sequence, filename
    except Exception:
        return None, None, filename

def process_language(language_name):
    """
    Procesa la alineación para un idioma en particular.
    Utiliza las variables definidas en la Celda 3:
      - tokenized_json_paths, processed_dirs, base_dir, languages, etc.
    """
    # Nombre en minúsculas (como clave en 'tokenized_json_paths') => 'italian', 'spanish', 'swahili'
    language_key_lower = language_name.lower()

    # Ruta al JSON tokenizado (Celda 3)
    transcripts_path = tokenized_json_paths[language_key_lower]

    # Ruta a la carpeta con los features preprocesados (Celda 3)
    features_dir = processed_dirs[language_name]

    # Construimos paths para guardar la alineación
    aligned_data_path = os.path.join(ALIGNED_DATA_DIR, f'aligned_data_{language_name}.npz')
    aligned_filenames_path = os.path.join(ALIGNED_DATA_DIR, f'aligned_filenames_{language_name}.txt')

    # Si no existe el JSON, lo creamos vacío (para no reventar)
    if not os.path.exists(transcripts_path):
        print(f"[ADVERTENCIA] {transcripts_path} no existe. Creando JSON vacío.")
        try:
            with open(transcripts_path, 'w', encoding='utf-8') as f:
                json.dump([], f, ensure_ascii=False)
        except Exception as e:
            print(f"No fue posible crear el JSON para {language_name}: {e}")
            return

    # Cargar datos existentes (si los hay)
    existing_mel = []
    existing_ids = []
    aligned_filenames = set()

    if os.path.exists(aligned_data_path):
        print(f"\nCargando datos alineados existentes para {language_name}...")
        try:
            data = np.load(aligned_data_path, allow_pickle=True)
            if 'mel_spectrograms' in data and 'input_ids' in data:
                existing_mel = list(data['mel_spectrograms'])
                existing_ids = list(data['input_ids'])
                if len(existing_mel) == len(existing_ids):
                    print(f"Datos existentes cargados: {len(existing_mel)} muestras.")
                else:
                    print("Los datos existentes están desbalanceados; continuaremos añadiendo.")
            else:
                print("No se encontraron las keys esperadas en el .npz. Procesando desde cero.")
        except Exception as e:
            print(f"Error al cargar datos alineados existentes: {e}. Procesando desde cero.")

    if os.path.exists(aligned_filenames_path):
        with open(aligned_filenames_path, 'r') as f:
            for line in f:
                aligned_filenames.add(line.strip())

    # Leer transcripciones del JSON
    try:
        with open(transcripts_path, 'r', encoding='utf-8') as f:
            transcripts = json.load(f)
    except Exception as e:
        print(f"Error al leer {transcripts_path} para {language_name}: {e}")
        return

    # Listas para muestras nuevas
    current_mel = []
    current_ids = []
    warning_log = []
    new_count = 0
    last_saved_count = 0

    # Procesamiento en paralelo
    futures = []
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        for entry in transcripts:
            filename = entry.get('audio_file', '')
            if filename in aligned_filenames:
                continue  # ya procesado
            futures.append(executor.submit(process_entry, entry, features_dir))

        for future in tqdm(futures, desc=f"Alineando {language_name}", total=len(futures)):
            mel, token_sequence, filename = future.result()
            if filename and (mel is None or token_sequence is None):
                warning_log.append(f"No se procesó correctamente {filename}")
                continue

            if mel is not None and token_sequence is not None and filename:
                current_mel.append(mel)
                current_ids.append(token_sequence)
                aligned_filenames.add(filename)
                new_count += 1

                # Guardado incremental cada SAVE_INTERVAL
                if new_count > last_saved_count and new_count % SAVE_INTERVAL == 0:
                    save_progress(
                        language_name,
                        aligned_data_path,
                        aligned_filenames_path,
                        existing_mel,
                        existing_ids,
                        current_mel,
                        current_ids,
                        aligned_filenames
                    )
                    last_saved_count = new_count

    # Guardado final
    if current_mel:
        save_progress(
            language_name,
            aligned_data_path,
            aligned_filenames_path,
            existing_mel,
            existing_ids,
            current_mel,
            current_ids,
            aligned_filenames
        )

    # Warnings
    if SHOW_WARNINGS and warning_log:
        print(f"\nSe encontraron {len(warning_log)} advertencias en {language_name}:")
        for msg in warning_log[:MAX_WARNINGS]:
            print(" -", msg)
        if len(warning_log) > MAX_WARNINGS:
            print(f" ... y {len(warning_log) - MAX_WARNINGS} más.")


# ----------------------------------------------
# Ejecutar la alineación para todos los idiomas
# ----------------------------------------------
for lang_name in languages.keys():
    process_language(lang_name)

print("\nProceso de alineación completado para todos los idiomas.")



Cargando datos alineados existentes para Italian...
Datos existentes cargados: 61058 muestras.


Alineando Italian: 100%|██████████| 795/795 [00:00<00:00, 9235.28it/s]



Se encontraron 795 advertencias en Italian:
 - No se procesó correctamente common_voice_it_17415772.wav
 - No se procesó correctamente common_voice_it_17415773.wav
 - No se procesó correctamente common_voice_it_17415774.wav
 - No se procesó correctamente common_voice_it_17415775.wav
 - No se procesó correctamente common_voice_it_17415776.wav
 - No se procesó correctamente common_voice_it_17415777.wav
 - No se procesó correctamente common_voice_it_17415778.wav
 - No se procesó correctamente common_voice_it_17415779.wav
 - No se procesó correctamente common_voice_it_17415780.wav
 - No se procesó correctamente common_voice_it_17415781.wav
 ... y 785 más.

Cargando datos alineados existentes para Spanish...
Datos existentes cargados: 62319 muestras.


Alineando Spanish: 0it [00:00, ?it/s]



Cargando datos alineados existentes para Swahili...
Datos existentes cargados: 61961 muestras.


Alineando Swahili: 100%|██████████| 4/4 [00:00<00:00,  4.58it/s]


Copia de seguridad creada: /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Swahili.npz.bak
Progreso guardado en /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Swahili.npz: 61965 muestras procesadas.

Proceso de alineación completado para todos los idiomas.


In [ ]:
# ===============================================
# Celda 5: Leer JSON con "text", entrenar subwords
# ===============================================

!pip install sentencepiece

import os
import json
import numpy as np
import sentencepiece as spm
from sklearn.model_selection import train_test_split

all_sentences_for_sp = []
splitted_data = {}

for lang_name in languages.keys():
    npz_path = os.path.join(ALIGNED_DATA_DIR, f"aligned_data_{lang_name}.npz")
    json_path = tokenized_json_paths[lang_name.lower()]

    if not os.path.exists(npz_path) or not os.path.exists(json_path):
        print(f"[ADVERTENCIA] Faltan archivos para {lang_name}.")
        splitted_data[lang_name] = ([], [], [], [])
        continue

    print(f"\nCargando {npz_path} y {json_path} para {lang_name}...")

    # 1) Cargar .npz
    npz_data = np.load(npz_path, allow_pickle=True)
    mel_list = list(npz_data["mel_spectrograms"])  # array de shape (N, 20,198)
    # Omitimos 'token_sequence_text' de .npz

    # 2) Cargar JSON con text
    with open(json_path, "r", encoding="utf-8") as f:
        transcripts = json.load(f)  # lista de dicts

    # Comprobar que mel_list y transcripts tengan misma longitud:
    if len(mel_list) != len(transcripts):
        print(f"[ADVERTENCIA] #mels {len(mel_list)} != #transcripts {len(transcripts)} => usar min(...)")
        min_len = min(len(mel_list), len(transcripts))
    else:
        min_len = len(mel_list)

    # 3) Emparejar (mel, text)
    mel_data  = []
    text_data = []
    for i in range(min_len):
        mel_data.append(mel_list[i])
        # tu JSON con "text" => transcripts[i]["text"]
        raw_text = transcripts[i]["text"]  # Ej: "it mostrare la luna nel pozzo."
        text_data.append(raw_text)
        all_sentences_for_sp.append(raw_text)

    # 4) Dividir train/val
    X_train, X_val, y_train_txt, y_val_txt = train_test_split(
        mel_data, text_data, test_size=0.1, random_state=42
    )
    splitted_data[lang_name] = (X_train, X_val, y_train_txt, y_val_txt)

# 5) Entrenar SentencePiece con all_sentences_for_sp
if len(all_sentences_for_sp) == 0:
    print("No hay transcripciones para entrenar subwords.")
else:
    with open("corpus.txt", "w", encoding="utf-8") as f:
        for line in all_sentences_for_sp:
            f.write(line.strip() + "\n")

    spm.SentencePieceTrainer.Train(
        input="corpus.txt",
        model_prefix="sp_unigram",
        vocab_size=8000,  # ajusta
        model_type="unigram",  # o "bpe"
        max_sentence_length=10000
    )
    print("SentencePiece entrenado.")
    sp = spm.SentencePieceProcessor(model_file="sp_unigram.model")

    # 6) Convertir y_train_txt, y_val_txt en subword IDs
    splitted_data_sub = {}
    for lang_name, (X_train, X_val, y_train_txt, y_val_txt) in splitted_data.items():
        if not X_train:  # sin datos
            splitted_data_sub[lang_name] = ([], [], [], [])
            continue

        y_train_ids = [sp.encode(t, out_type=int) for t in y_train_txt]
        y_val_ids   = [sp.encode(t, out_type=int) for t in y_val_txt]

        splitted_data_sub[lang_name] = (X_train, X_val, y_train_ids, y_val_ids)

    splitted_data = splitted_data_sub
    print("\nAhora splitted_data tiene subword IDs en y_train, y_val.")



Cargando /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Italian.npz y /content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/transcriptionsitTok.json para Italian...
[ADVERTENCIA] #mels 61058 != #transcripts 61853 => usar min(...)

Cargando /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Spanish.npz y /content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/transcriptionsesTok.json para Spanish...

Cargando /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Swahili.npz y /content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/transcriptionsswTok.json para Swahili...
SentencePiece entrenado.

Ahora splitted_data tiene subword IDs en y_train, y_val.


In [ ]:
# ===============================================
# Celda 6: Unificar multilingüe y preprocesar (20 frames, 20 tokens)
# ===============================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from scipy.interpolate import interp1d

X_train_all = []
y_train_all = []
X_val_all   = []
y_val_all   = []

# Recorremos splitted_data (generado en Celda 5 con subword IDs)
for lang_name, (X_train, X_val, y_train_ids, y_val_ids) in splitted_data.items():
    if len(X_train) == 0:
        print(f"[ADVERTENCIA] {lang_name} no tiene datos => se omite")
        continue
    X_train_all.extend(X_train)
    y_train_all.extend(y_train_ids)
    X_val_all.extend(X_val)
    y_val_all.extend(y_val_ids)

print(f"\nTotal muestras en entrenamiento (todos idiomas): {len(X_train_all)}")
print(f"Total muestras en validación (todos idiomas):    {len(X_val_all)}")

if len(X_train_all) == 0:
    print("No hay datos unificados. Revisa celdas anteriores.")
else:
    # Forzamos que la longitud de salida sea 20 tokens para que coincida con la cantidad de frames
    max_length = 20
    print(f"-> Forzamos max_length (tokens) = {max_length}.")

    # Configuración para espectrogramas: vamos a usar 20 frames
    n_mels = 128
    max_spectrogram_length = 20

    def pad_or_truncate_spectrogram(spec, max_time, n_mels):
        # Ajuste en la dimensión de frecuencia: interpola a n_mels columnas si es necesario
        if spec.shape[1] != n_mels:
            x = np.arange(spec.shape[1])
            f = interp1d(x, spec, axis=1, kind='linear')
            spec = f(np.linspace(0, spec.shape[1] - 1, n_mels))
        # Ajuste en la dimensión de tiempo: se trunca o se le hace padding para forzar max_time frames
        time_steps = spec.shape[0]
        if time_steps < max_time:
            pad_width = max_time - time_steps
            spec = np.pad(spec, ((0, pad_width), (0, 0)), mode='constant')
        else:
            spec = spec[:max_time, :]
        return spec

    # Preprocesar los espectrogramas
    X_train_padded = np.array([pad_or_truncate_spectrogram(s, max_spectrogram_length, n_mels)
                                 for s in X_train_all])
    X_val_padded = np.array([pad_or_truncate_spectrogram(s, max_spectrogram_length, n_mels)
                               for s in X_val_all])

    # Preprocesar las secuencias de subword (tokens)
    y_train_padded = pad_sequences(y_train_all, maxlen=max_length, padding='post', truncating='post')
    y_val_padded = pad_sequences(y_val_all, maxlen=max_length, padding='post', truncating='post')

    # Usar generator para evitar cargar un tensor enorme a la GPU de golpe
    def gen_train():
        for x, y in zip(X_train_padded, y_train_padded):
            yield x.astype(np.float32), y.astype(np.int32)

    def gen_val():
        for x, y in zip(X_val_padded, y_val_padded):
            yield x.astype(np.float32), y.astype(np.int32)

    train_dataset = tf.data.Dataset.from_generator(
        gen_train,
        output_signature=(
            tf.TensorSpec(shape=(max_spectrogram_length, n_mels), dtype=tf.float32),
            tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
        )
    )
    train_dataset = train_dataset.shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_generator(
        gen_val,
        output_signature=(
            tf.TensorSpec(shape=(max_spectrogram_length, n_mels), dtype=tf.float32),
            tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
        )
    )
    val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

    print("Datasets multilingües creados (20 frames, 20 tokens):")
    print(f" - train_dataset con {len(X_train_all)} muestras")
    print(f" - val_dataset con {len(X_val_all)} muestras")



Total muestras en entrenamiento (todos idiomas): 166807
Total muestras en validación (todos idiomas):    18535
-> Forzamos max_length (tokens) = 20.
Datasets multilingües creados (20 frames, 20 tokens):
 - train_dataset con 166807 muestras
 - val_dataset con 18535 muestras


In [ ]:
# ===============================================
# Celda 7: Definir/cargar modelo subwords multilingüe (20 frames, 20 tokens, LR=1e-4)
# ===============================================

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

if len(X_train_all) == 0:
    print("No hay datos unificados => no se define modelo.")
else:
    all_tokens = np.concatenate(y_train_padded)
    vocab_size = int(np.max(all_tokens)) + 1
    print(f"Vocab subword size = {vocab_size}")

    multiling_model_path = os.path.join(checkpoint_dir, "best_model_multiling_subwords.h5")

    class BahdanauAttention(tf.keras.layers.Layer):
        def __init__(self, units, **kwargs):
            super().__init__(**kwargs)
            self.units = units
            self.W1 = layers.Dense(units)
            self.W2 = layers.Dense(units)
            self.V  = layers.Dense(1)

        def get_config(self):
            config = super().get_config()
            config.update({"units": self.units})
            return config

        def call(self, query, values):
            query_with_time_axis = tf.expand_dims(query, 1)
            score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
            attention_weights = tf.nn.softmax(score, axis=1)
            context_vector = attention_weights * values
            context_vector = tf.reduce_sum(context_vector, axis=1)
            return context_vector, attention_weights

    # Función para forzar que la salida del encoder tenga max_length pasos (20)
    def pad_encoder_outputs(x, max_time=20):
        import tensorflow as tf
        actual_time = tf.shape(x)[1]
        pad_needed = max_time - actual_time
        x_padded = tf.cond(
            pad_needed > 0,
            lambda: tf.pad(x, [[0, 0], [0, pad_needed], [0, 0]], mode="CONSTANT"),
            lambda: x[:, :max_time, :]
        )
        return x_padded

    def create_multiling_model(input_shape, vocab_size, units=256, max_length=20):
        inputs = layers.Input(shape=input_shape)  # input_shape: (20, 128)
        x = layers.Masking(mask_value=0.0)(inputs)
        x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        x = layers.Dropout(0.2)(x)
        encoder_outputs = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        encoder_outputs = layers.Dropout(0.2)(encoder_outputs)

        # Forzar encoder_outputs a tener 20 pasos
        encoder_outputs = layers.Lambda(lambda t: pad_encoder_outputs(t, max_time=max_length))(encoder_outputs)

        attention_layer = BahdanauAttention(units)
        query = encoder_outputs[:, -1, :]  # (batch, 2*units)
        context_vector, attention_weights = attention_layer(query, encoder_outputs)

        context_vector_rep = layers.RepeatVector(max_length)(context_vector)  # (batch, 20, 2*units)
        decoder_input = layers.Concatenate(axis=-1)([encoder_outputs, context_vector_rep])
        # (batch, 20, 4*units)

        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(decoder_input)
        x = layers.Dropout(0.2)(x)
        outputs = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)

        model = models.Model(inputs=inputs, outputs=outputs)
        return model

    # input_shape = (max_spectrogram_length, n_mels) = (20, 128)
    input_shape = (max_spectrogram_length, n_mels)

    if os.path.exists(multiling_model_path):
        print(f"Cargando modelo existente desde {multiling_model_path}")
        model = tf.keras.models.load_model(multiling_model_path,
                                           custom_objects={'BahdanauAttention': BahdanauAttention})
    else:
        print("Creando un nuevo modelo subwords multilingüe (20 frames, 20 tokens)...")
        model = create_multiling_model(input_shape, vocab_size, units=256, max_length=max_length)
        model.compile(
            optimizer=optimizers.Adam(learning_rate=1e-4),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

    print(model.summary())


Vocab subword size = 8000
Cargando modelo existente desde /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/checkpoints/best_model_multiling_subwords.h5
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 20, 128)]    0           []                               
                                                                                                  
 masking_1 (Masking)            (None, 20, 128)      0           ['input_2[0][0]']                
                                                                                                  
 conv1d_1 (Conv1D)              (None, 20, 128)      49280       ['masking_1[0][0]']              
                                                                                                  
 dropout_3 (Dropout)

In [ ]:
# ===============================================
# Celda 8: Entrenar el modelo subwords multilingüe (20 frames, 20 tokens)
# ===============================================

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard, ReduceLROnPlateau

if len(X_train_all) == 0:
    print("No hay datos de entrenamiento => no se entrena.")
else:
    model_name = "best_model_multiling_subwords.h5"
    checkpoint_cb = ModelCheckpoint(
        filepath=os.path.join(checkpoint_dir, model_name),
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        mode="min"
    )
    early_stopping_cb = EarlyStopping(
        monitor="val_loss",
        patience=10,
        verbose=1,
        restore_best_weights=True
    )
    tensorboard_cb = TensorBoard(log_dir=logs_dir)
    lr_scheduler_cb = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        verbose=1,
        mode="min"
    )

    epochs = 100
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=[checkpoint_cb, early_stopping_cb, tensorboard_cb, lr_scheduler_cb]
    )

    print("Entrenamiento multilingüe (subwords) completado.")


Epoch 1/100
   5213/Unknown - 224s 43ms/step - loss: 4.4477 - accuracy: 0.3693
Epoch 1: val_loss improved from inf to 5.86638, saving model to /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/checkpoints/best_model_multiling_subwords.h5
5213/5213 [==============================] - 232s 44ms/step - loss: 4.4477 - accuracy: 0.3693 - val_loss: 5.8664 - val_accuracy: 0.3266 - lr: 2.5000e-06
Epoch 2/100
5213/5213 [==============================] - ETA: 0s - loss: 4.4583 - accuracy: 0.3690
Epoch 2: val_loss improved from 5.86638 to 5.86125, saving model to /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/checkpoints/best_model_multiling_subwords.h5
5213/5213 [==============================] - 234s 45ms/step - loss: 4.4583 - accuracy: 0.3690 - val_loss: 5.8613 - val_accuracy: 0.3267 - lr: 2.5000e-06
Epoch 3/100
5213/5213 [==============================] - ETA: 0s - loss: 4.4592 - accuracy: 0.3689
Epoch 3: val_loss improved from 5.86125 to 5.85397, sav

In [ ]:
# ===============================================
# Celda 5: Carga y separación de datos (train_test_split)
# ===============================================

import os
import numpy as np
from sklearn.model_selection import train_test_split

# Aquí se guardan los splits para cada idioma.
# Formato: splitted_data["Italian"] = (X_train, X_val, y_train, y_val), etc.
splitted_data = {}

# Para cada idioma en el diccionario 'languages'
for lang_name in languages.keys():
    # Construir la ruta al .npz que generó la Celda 4
    aligned_data_path = os.path.join(ALIGNED_DATA_DIR, f"aligned_data_{lang_name}.npz")

    if not os.path.exists(aligned_data_path):
        print(f"[ADVERTENCIA] No se encontró {aligned_data_path}. Revisa la Celda 4 para {lang_name}.")
        # Guardamos tuplas vacías para no romper celdas posteriores
        splitted_data[lang_name] = ([], [], [], [])
        continue

    try:
        print(f"\nCargando datos desde {aligned_data_path} para {lang_name}...")
        data = np.load(aligned_data_path, allow_pickle=True)
        mel_spectrograms = list(data.get("mel_spectrograms", []))
        input_ids = list(data.get("input_ids", []))

        # Verificar si están vacíos
        if not mel_spectrograms or not input_ids:
            print(f"El archivo {aligned_data_path} está vacío. Revisa la Celda 4 para {lang_name}.")
            splitted_data[lang_name] = ([], [], [], [])
            continue

        # Separar en entrenamiento y validación (por ejemplo, 90% / 10%)
        X_train, X_val, y_train, y_val = train_test_split(
            mel_spectrograms,
            input_ids,
            test_size=0.1,
            random_state=42
        )

        print(f"[{lang_name}] Datos de entrenamiento: {len(X_train)} muestras")
        print(f"[{lang_name}] Datos de validación: {len(X_val)} muestras")

        # Almacenar el split en nuestro diccionario
        splitted_data[lang_name] = (X_train, X_val, y_train, y_val)

    except Exception as e:
        print(f"[ERROR] Al cargar {aligned_data_path} para {lang_name}: {e}")
        splitted_data[lang_name] = ([], [], [], [])

print("\nProceso de carga y separación completado para todos los idiomas.")



Cargando datos desde /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Italian.npz para Italian...
[ERROR] Al cargar /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Italian.npz para Italian: sequence item 0: expected str instance, int found

Cargando datos desde /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Spanish.npz para Spanish...
[ERROR] Al cargar /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Spanish.npz para Spanish: sequence item 0: expected str instance, int found

Cargando datos desde /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Swahili.npz para Swahili...
[ERROR] Al cargar /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data/aligned_data_Swahili.npz para Swahili: sequence item 0: expected str instance, in

RuntimeError: Internal: src/trainer_interface.cc(431) [!sentences_.empty()] 

In [ ]:
# ===============================================
# Celda 6: Unificar y preprocesar datos (multilingüe)
# ===============================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from scipy.interpolate import interp1d

# 1. Unificamos los datos de splitted_data
X_train_all = []
y_train_all = []
X_val_all = []
y_val_all = []

for lang_name, (X_train, X_val, y_train, y_val) in splitted_data.items():
    if len(X_train) == 0 and len(X_val) == 0:
        print(f"[ADVERTENCIA] {lang_name} no tiene datos. Se omite en la fusión.")
    else:
        # Unir a las listas globales
        X_train_all.extend(X_train)
        y_train_all.extend(y_train)
        X_val_all.extend(X_val)
        y_val_all.extend(y_val)

print(f"\nTotal de muestras en entrenamiento (todos los idiomas): {len(X_train_all)}")
print(f"Total de muestras en validación (todos los idiomas): {len(X_val_all)}")

if len(X_train_all) == 0:
    print("No hay datos de entrenamiento unificados. Revisa la Celda 5.")
else:
    # 2. Calcular la longitud máxima de secuencia (tokens) en todo el train (p.e. percentil 95)
    train_seq_lengths = [len(seq) for seq in y_train_all]
    percentil = 95
    max_length = int(np.percentile(train_seq_lengths, percentil))
    print(f"Longitud máxima de tokens (p{percentil}): {max_length}")

    # 3. Configuración de los espectrogramas
    n_mels = 128
    max_spectrogram_length = 20

    def pad_or_truncate_spectrogram(spectrogram, max_time, n_mels):
        """Redimensionar un espectrograma a (max_time, n_mels)."""
        # Interpolación en freq
        if spectrogram.shape[1] != n_mels:
            x = np.arange(spectrogram.shape[1])
            f = interp1d(x, spectrogram, axis=1, kind='linear')
            spectrogram = f(np.linspace(0, spectrogram.shape[1] - 1, n_mels))

        # Pad / truncate en tiempo
        time_steps = spectrogram.shape[0]
        if time_steps < max_time:
            pad_width = max_time - time_steps
            spectrogram = np.pad(spectrogram, ((0, pad_width), (0, 0)), mode='constant')
        else:
            spectrogram = spectrogram[:max_time, :]
        return spectrogram

    # 4. Aplicar la función a todo el train y val
    X_train_padded = np.array([
        pad_or_truncate_spectrogram(s, max_spectrogram_length, n_mels)
        for s in X_train_all
    ])
    X_val_padded = np.array([
        pad_or_truncate_spectrogram(s, max_spectrogram_length, n_mels)
        for s in X_val_all
    ])

    # Pad de las secuencias de tokens
    y_train_padded = pad_sequences(
        y_train_all, maxlen=max_length, padding='post', truncating='post'
    )
    y_val_padded = pad_sequences(
        y_val_all, maxlen=max_length, padding='post', truncating='post'
    )

    # 5. Crear dataset de TensorFlow
    batch_size = 32
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train_padded, y_train_padded))
    train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val_padded, y_val_padded))
    val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    print("Datasets multilingües preparados:")
    print(f" - train_dataset con {len(X_train_all)} muestras")
    print(f" - val_dataset con {len(X_val_all)} muestras")




Total de muestras en entrenamiento (todos los idiomas): 166803
Total de muestras en validación (todos los idiomas): 18535
Longitud máxima de tokens (p95): 15
Datasets multilingües preparados:
 - train_dataset con 166803 muestras
 - val_dataset con 18535 muestras


In [ ]:
# ===============================================
# Celda 7: Definir o cargar el modelo multilingüe (corregida)
# ===============================================

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

if len(X_train_all) == 0:
    print("No hay datos unificados para entrenar. No se define modelo.")
else:
    # Calcular vocab_size a partir de y_train_padded (definidos en Celda 6)
    all_tokens = np.concatenate(y_train_padded)
    vocab_size = int(np.max(all_tokens)) + 1
    print(f"Tamaño del vocabulario multilingüe: {vocab_size}")

    # Ruta del modelo (solo uno, para todos los idiomas)
    multiling_model_path = os.path.join(checkpoint_dir, "best_model_multilingue.h5")

    class BahdanauAttention(tf.keras.layers.Layer):
        def __init__(self, units, **kwargs):
            super(BahdanauAttention, self).__init__(**kwargs)
            self.units = units
            self.W1 = layers.Dense(units)
            self.W2 = layers.Dense(units)
            self.V = layers.Dense(1)

        def get_config(self):
            # Sobrescribimos para poder serializar la capa sin warning
            config = super(BahdanauAttention, self).get_config()
            config.update({
                "units": self.units
            })
            return config

        def call(self, query, values):
            # query shape: (batch, units)
            query_with_time_axis = tf.expand_dims(query, 1)  # (batch, 1, units)
            score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
            attention_weights = tf.nn.softmax(score, axis=1)  # (batch, time, 1)
            context_vector = attention_weights * values       # (batch, time, units)
            context_vector = tf.reduce_sum(context_vector, axis=1)  # (batch, units)
            return context_vector, attention_weights

    def create_multiling_model(input_shape, vocab_size, units=256, max_length=15):
        inputs = layers.Input(shape=input_shape)
        x = layers.Masking(mask_value=0.0)(inputs)
        x = layers.Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        x = layers.Dropout(0.2)(x)
        encoder_outputs = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        encoder_outputs = layers.Dropout(0.2)(encoder_outputs)

        # Forzamos la dimensión de tiempo del encoder a 'max_length'
        # para que coincida al concatenar con RepeatVector
        encoder_outputs = layers.Lambda(lambda t: t[:, :max_length, :])(encoder_outputs)

        attention_layer = BahdanauAttention(units)
        query = encoder_outputs[:, -1, :]  # (batch, units*2)
        context_vector, attention_weights = attention_layer(query, encoder_outputs)

        # Repetir el contexto 'max_length' veces
        context_vector_repeated = layers.RepeatVector(max_length)(context_vector)

        # Concat dimension => (batch, max_length, units*4) si Bidirectional * 2
        decoder_input = layers.Concatenate(axis=-1)([encoder_outputs, context_vector_repeated])

        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(decoder_input)
        x = layers.Dropout(0.2)(x)
        outputs = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)

        model = models.Model(inputs=inputs, outputs=outputs)
        return model

    input_shape = (max_spectrogram_length, n_mels)

    # Si existe el modelo entrenado, lo cargamos
    if os.path.exists(multiling_model_path):
        print(f"Cargando modelo multilingüe existente desde {multiling_model_path}...")
        model = tf.keras.models.load_model(multiling_model_path)
    else:
        print("Creando un nuevo modelo multilingüe...")
        model = create_multiling_model(input_shape, vocab_size, units=256, max_length=max_length)
        model.compile(
            optimizer=optimizers.Adam(learning_rate=1e-5),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

    print(model.summary())


Tamaño del vocabulario multilingüe: 168620
Creando un nuevo modelo multilingüe...
Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 20, 128)]    0           []                               
                                                                                                  
 masking_2 (Masking)            (None, 20, 128)      0           ['input_3[0][0]']                
                                                                                                  
 conv1d_2 (Conv1D)              (None, 20, 128)      49280       ['masking_2[0][0]']              
                                                                                                  
 dropout_7 (Dropout)            (None, 20, 128)      0           ['conv1d_2[0][0]']               
          

In [ ]:
# ===============================================
# Celda 8: Entrenar el modelo multilingüe
# ===============================================

import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard

if len(X_train_all) == 0:
    print("No hay datos unificados para entrenar. Se omite el entrenamiento.")
else:
    model_name = "best_model_multilingue.h5"
    checkpoint_cb = ModelCheckpoint(
        filepath=os.path.join(checkpoint_dir, model_name),
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        mode="min"
    )

    early_stopping_cb = EarlyStopping(
        monitor="val_loss",
        patience=5,
        verbose=1,
        restore_best_weights=True
    )

    tensorboard_cb = TensorBoard(log_dir=logs_dir)

    epochs = 15
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=[checkpoint_cb, early_stopping_cb, tensorboard_cb]
    )

    print("Entrenamiento multilingüe completado.")

    # (Opcional) Guardar también un "final_model_multilingue.h5"
    # model.save(os.path.join(checkpoint_dir, "final_model_multilingue.h5"))


Epoch 1/15
5213/5213 [==============================] - ETA: 0s - loss: 6.5594 - accuracy: 0.3580
Epoch 1: val_loss improved from inf to 6.02005, saving model to /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/checkpoints/best_model_multilingue.h5
5213/5213 [==============================] - 1237s 235ms/step - loss: 6.5594 - accuracy: 0.3580 - val_loss: 6.0200 - val_accuracy: 0.3750
Epoch 2/15
5213/5213 [==============================] - ETA: 0s - loss: 5.1149 - accuracy: 0.4145
Epoch 2: val_loss improved from 6.02005 to 5.95223, saving model to /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/checkpoints/best_model_multilingue.h5
5213/5213 [==============================] - 1257s 241ms/step - loss: 5.1149 - accuracy: 0.4145 - val_loss: 5.9522 - val_accuracy: 0.3821
Epoch 3/15
5213/5213 [==============================] - ETA: 0s - loss: 4.9202 - accuracy: 0.4263
Epoch 3: val_loss did not improve from 5.95223
5213/5213 [========================

In [ ]:
# ===============================================
# VIEJO Celda 4: Alinear los audios
# ===============================================

import os
import numpy as np
import json
from tqdm import tqdm

# Configuración
TARGET_SHAPE = (20, 198)  # Dimensión deseada
SHOW_WARNINGS = True
MAX_WARNINGS = 10
SAVE_INTERVAL = 1000

# Variables globales
aligned_mel_spectrograms = []  # Acumula los espectrogramas alineados de todos los idiomas
aligned_input_ids = []         # Acumula las secuencias alineadas de todos los idiomas

def resize_mel(mel, target_shape):
    """Redimensiona el espectrograma Mel a una forma fija."""
    current_shape = mel.shape
    if current_shape[0] > target_shape[0]:  # Recortar filas
        mel = mel[:target_shape[0], :]
    elif current_shape[0] < target_shape[0]:  # Añadir filas con ceros (padding)
        padding = np.zeros((target_shape[0] - current_shape[0], current_shape[1]))
        mel = np.vstack([mel, padding])

    if current_shape[1] > target_shape[1]:  # Recortar columnas
        mel = mel[:, :target_shape[1]]
    elif current_shape[1] < target_shape[1]:  # Añadir columnas con ceros
        padding = np.zeros((mel.shape[0], target_shape[1] - current_shape[1]))
        mel = np.hstack([mel, padding])

    return mel

# Procesar idiomas
for language_name, language_code in languages.items():
    aligned_data_path = os.path.join(base_dir, f'aligned_data_{language_name}.npz')
    transcripts_path = os.path.join('/content/drive/My Drive/TinyML for Translation/SST & TTS',
                                    language_name, f'transcriptions{language_code}Tok.json')
    features_dir = processed_dirs[language_name]

    # Verificar existencia de transcripciones y directorios
    if not os.path.exists(transcripts_path):
        print(f"Transcripciones no encontradas para {language_name}. Saltando...")
        continue
    if not os.path.exists(features_dir):
        print(f"Directorio de características no encontrado para {language_name}. Saltando...")
        continue

    # Cargar datos existentes y saltar si ya están completos
    if os.path.exists(aligned_data_path):
        print(f"\nCargando datos alineados existentes para {language_name}...")
        try:
            data = np.load(aligned_data_path, allow_pickle=True)
            existing_mel = list(data['mel_spectrograms'])
            existing_ids = list(data['input_ids'])

            if len(existing_mel) > 0 and len(existing_mel) == len(existing_ids):
                # Acumular datos ya alineados globalmente y pasar al siguiente idioma
                aligned_mel_spectrograms.extend(existing_mel)
                aligned_input_ids.extend(existing_ids)
                print(f"Datos existentes cargados: {len(existing_mel)} muestras. Saltando alineación.")
                continue
            else:
                print(f"Datos incompletos para {language_name}. Reanudando alineación.")
        except Exception as e:
            print(f"Error al cargar datos alineados existentes: {e}. Reanudando alineación.")

    # Cargar transcripciones
    transcripts = []
    try:
        with open(transcripts_path, 'r', encoding='utf-8') as f:
            transcripts = json.load(f)
    except Exception as e:
        print(f"Error al cargar transcripciones para {language_name}: {e}")
        continue

    # Inicializar listas y variables
    current_mel = []
    current_ids = []
    warning_log = []
    aligned_filenames = set()

    # Registrar archivos alineados existentes
    aligned_filenames_path = os.path.join(base_dir, f'aligned_filenames_{language_name}.txt')
    if os.path.exists(aligned_filenames_path):
        with open(aligned_filenames_path, 'r') as f:
            aligned_filenames = set(line.strip() for line in f)

    # Alinear nuevos datos
    new_count = 0
    last_saved_count = 0

    for idx, entry in enumerate(tqdm(transcripts, desc=f"Alineando {language_name}", total=len(transcripts))):
        filename = entry.get('audio_file', '')
        token_sequence = entry.get('token_sequence_text', [])
        if not filename or not token_sequence or filename in aligned_filenames:
            continue

        npy_filename = os.path.splitext(filename)[0] + '.npy'
        feature_path = os.path.join(features_dir, npy_filename)

        if os.path.exists(feature_path):
            try:
                mel = np.load(feature_path, allow_pickle=True)
                mel = resize_mel(mel, TARGET_SHAPE)  # Redimensionar espectrograma
                current_mel.append(mel)
                current_ids.append(token_sequence)
                aligned_filenames.add(filename)
                new_count += 1
            except Exception as e:
                warning_log.append(f"Error al cargar {feature_path}: {e}")
        else:
            warning_log.append(f"{feature_path} no encontrado")

        # Guardar progreso incremental
        if new_count > last_saved_count and new_count % SAVE_INTERVAL == 0:
            try:
                mel_array = np.array(current_mel, dtype=object)
                ids_array = np.array(current_ids, dtype=object)
                np.savez(aligned_data_path, mel_spectrograms=mel_array, input_ids=ids_array)
                with open(aligned_filenames_path, 'w') as f:
                    for fn in aligned_filenames:
                        f.write(fn + '\n')
                print(f"Progreso guardado: {new_count} muestras procesadas.")
                last_saved_count = new_count
            except Exception as e:
                print(f"Error al guardar progreso incremental: {e}")

    # Guardar datos alineados válidos al finalizar
    try:
        mel_array = np.array(current_mel, dtype=object)
        ids_array = np.array(current_ids, dtype=object)
        np.savez(aligned_data_path, mel_spectrograms=mel_array, input_ids=ids_array)
        with open(aligned_filenames_path, 'w') as f:
            for fn in aligned_filenames:
                f.write(fn + '\n')
        print(f"Datos alineados guardados para {language_name}: {len(current_mel)} muestras (nuevas: {new_count}).")
    except Exception as e:
        print(f"Error al guardar datos alineados finales para {language_name}: {e}")

    # Acumular resultados globales
    aligned_mel_spectrograms.extend(current_mel)
    aligned_input_ids.extend(current_ids)

    # Mostrar advertencias limitadas
    if SHOW_WARNINGS and warning_log:
        print(f"\nSe encontraron {len(warning_log)} advertencias en {language_name}. Mostrando las primeras {MAX_WARNINGS}:")
        for msg in warning_log[:MAX_WARNINGS]:
            print(f" - {msg}")
        if len(warning_log) > MAX_WARNINGS:
            print(f" ... y {len(warning_log) - MAX_WARNINGS} más.")

# Resumen final
print(f"\nProceso completado. Total de muestras alineadas: {len(aligned_mel_spectrograms)}")



In [ ]:
# Celda 4.1: Liberar memoria después de la alineación
import gc

# Liberar variables grandes de memoria
aligned_mel_spectrograms = None
aligned_input_ids = None

# Forzar recolección de basura
gc.collect()

print("Memoria RAM liberada después de la alineación.")


Memoria RAM liberada después de la alineación.


In [ ]:
# Celda 5: Dividir datos en conjuntos de entrenamiento y validación
from sklearn.model_selection import train_test_split

SAVE_PATH = "aligned_data_global.npz"

# Verificar la existencia del archivo de datos alineados
if not os.path.exists(SAVE_PATH):
    print(f"No se encontró el archivo de datos alineados: {SAVE_PATH}. Revisa la Celda 4.")
else:
    try:
        # Intentar cargar los datos alineados
        print(f"Cargando datos desde {SAVE_PATH}...")
        data = np.load(SAVE_PATH, allow_pickle=True)
        aligned_mel_spectrograms = list(data.get("mel_spectrograms", []))
        aligned_input_ids = list(data.get("input_ids", []))

        # Verificar si los datos están vacíos
        if not aligned_mel_spectrograms or not aligned_input_ids:
            print("El archivo de datos alineados está vacío. Revisa la Celda 4.")
        else:
            # Dividir los datos en conjuntos de entrenamiento y validación
            X_train, X_val, y_train, y_val = train_test_split(
                aligned_mel_spectrograms,
                aligned_input_ids,
                test_size=0.1,
                random_state=42
            )

            print(f"Datos de entrenamiento: {len(X_train)} muestras")
            print(f"Datos de validación: {len(X_val)} muestras")
    except Exception as e:
        print(f"Error al cargar el archivo: {e}. Revisa la Celda 4.")


Cargando datos desde aligned_data_global.npz...
Error al cargar el archivo: No data left in file. Revisa la Celda 4.


In [ ]:
# Celda 6: Calcular max_length y preparar datasets

if len(X_train) > 0:
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from scipy.interpolate import interp1d

    # Calcular longitud máxima de las secuencias (percentil 95)
    train_seq_lengths = [len(seq) for seq in y_train]
    percentil = 95
    max_length_train = int(np.percentile(train_seq_lengths, percentil))
    max_length = max_length_train
    print(f"Longitud máxima (95%): {max_length}")

    # Configuración de los espectrogramas
    n_mels = 128
    max_spectrogram_length = 20

    def pad_or_truncate_spectrogram(spectrogram, max_length):
        """Redimensionar espectrogramas a dimensiones fijas."""
        if spectrogram.shape[1] != n_mels:
            x = np.arange(spectrogram.shape[1])
            f = interp1d(x, spectrogram, axis=1, kind='linear')
            spectrogram = f(np.linspace(0, spectrogram.shape[1] - 1, n_mels))

        time_steps = spectrogram.shape[0]
        if time_steps < max_length:
            pad_width = max_length - time_steps
            padded_spectrogram = np.pad(spectrogram, ((0, pad_width), (0, 0)), mode='constant')
        else:
            padded_spectrogram = spectrogram[:max_length, :]
        return padded_spectrogram

    # Preparar datos
    X_train_padded = np.array([pad_or_truncate_spectrogram(s, max_spectrogram_length) for s in X_train])
    X_val_padded = np.array([pad_or_truncate_spectrogram(s, max_spectrogram_length) for s in X_val])
    y_train_padded = pad_sequences(y_train, maxlen=max_length, padding='post', truncating='post')
    y_val_padded = pad_sequences(y_val, maxlen=max_length, padding='post', truncating='post')

    # Crear datasets de TensorFlow
    batch_size = 32
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train_padded, y_train_padded))
    train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val_padded, y_val_padded))
    val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    print(f"Datasets preparados con batch size = {batch_size}.")
else:
    print("No hay datos disponibles para preparar los datasets.")


Longitud máxima (95%): 15


In [ ]:
# PUEDE QUE SEA LA BUENA EN VEZ DE LA SIGUIENTE...Celda 7: Definir el modelo y entrenar

if len(aligned_mel_spectrograms) > 0:
    all_tokens = np.concatenate(aligned_input_ids)
    vocab_size = int(np.max(all_tokens)) + 1
    print(f"Tamaño del vocabulario: {vocab_size}")

    class BahdanauAttention(tf.keras.layers.Layer):
        def __init__(self, units):
            super(BahdanauAttention, self).__init__()
            self.W1 = layers.Dense(units)
            self.W2 = layers.Dense(units)
            self.V = layers.Dense(1)

        def call(self, query, values):
            query_with_time_axis = tf.expand_dims(query, 1)
            score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
            attention_weights = tf.nn.softmax(score, axis=1)
            context_vector = attention_weights * values
            context_vector = tf.reduce_sum(context_vector, axis=1)
            return context_vector, attention_weights

    def create_model(input_shape, vocab_size, units=256, max_length=15):
        inputs = layers.Input(shape=input_shape)
        x = layers.Masking(mask_value=0.0)(inputs)
        x = layers.Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        x = layers.Dropout(0.2)(x)
        encoder_outputs = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        encoder_outputs = layers.Dropout(0.2)(encoder_outputs)

        attention_layer = BahdanauAttention(units)
        query = encoder_outputs[:, -1, :]
        context_vector, attention_weights = attention_layer(query, encoder_outputs)

        context_vector_repeated = layers.RepeatVector(max_length)(context_vector)
        decoder_input = layers.Concatenate(axis=-1)([encoder_outputs, context_vector_repeated])

        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(decoder_input)
        x = layers.Dropout(0.2)(x)
        outputs = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)

        model = models.Model(inputs=inputs, outputs=outputs)
        return model

    input_shape = (max_length, n_mels)
    model = create_model(input_shape, vocab_size, units=256, max_length=max_length)

    model.compile(optimizer=optimizers.Adam(learning_rate=1e-5), loss='sparse_categorical_crossentropy')

    # Definir los callbacks
    checkpoint_cb = ModelCheckpoint(
        filepath=os.path.join(checkpoint_dir, 'model_epoch_{epoch:02d}_val_loss_{val_loss:.4f}.h5'),
        monitor='val_loss',
        verbose=1,
        save_best_only=True,
        mode='min'
    )

    early_stopping_cb = EarlyStopping(
        monitor='val_loss',
        patience=5,
        verbose=1,
        restore_best_weights=True
    )

    tensorboard_cb = TensorBoard(log_dir=logs_dir)

    # Entrenar el modelo
    epochs = 15
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=[checkpoint_cb, early_stopping_cb, tensorboard_cb]  # Puedes agregar wer_cer_callback si lo tienes
    )

    print("Entrenamiento completado. El modelo ha sido guardado durante el entrenamiento en los checkpoints.")
else:
    print("No hay datos para entrenar el modelo.")


In [ ]:
# Celda 7: Definir el modelo o cargar modelo existente
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard

# Tamaño del vocabulario
all_tokens = np.concatenate(y_train_padded)
vocab_size = int(np.max(all_tokens)) + 1
print(f"Tamaño del vocabulario: {vocab_size}")

# Verificar si existe un modelo guardado
model_path = os.path.join(checkpoint_dir, "best_model.h5")

if os.path.exists(model_path):
    print("Cargando modelo existente...")
    model = tf.keras.models.load_model(model_path)
else:
    print("Creando un nuevo modelo...")

    # Crear modelo con Bahdanau Attention
    class BahdanauAttention(tf.keras.layers.Layer):
        def __init__(self, units):
            super(BahdanauAttention, self).__init__()
            self.W1 = layers.Dense(units)
            self.W2 = layers.Dense(units)
            self.V = layers.Dense(1)

        def call(self, query, values):
            query_with_time_axis = tf.expand_dims(query, 1)
            score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
            attention_weights = tf.nn.softmax(score, axis=1)
            context_vector = attention_weights * values
            context_vector = tf.reduce_sum(context_vector, axis=1)
            return context_vector, attention_weights

    def create_model(input_shape, vocab_size, units=256, max_length=15):
        inputs = layers.Input(shape=input_shape)
        x = layers.Masking(mask_value=0.0)(inputs)
        x = layers.Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        x = layers.Dropout(0.2)(x)
        encoder_outputs = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
        encoder_outputs = layers.Dropout(0.2)(encoder_outputs)

        attention_layer = BahdanauAttention(units)
        query = encoder_outputs[:, -1, :]
        context_vector, attention_weights = attention_layer(query, encoder_outputs)

        context_vector_repeated = layers.RepeatVector(max_length)(context_vector)
        decoder_input = layers.Concatenate(axis=-1)([encoder_outputs, context_vector_repeated])

        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(decoder_input)
        x = layers.Dropout(0.2)(x)
        outputs = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)

        model = models.Model(inputs=inputs, outputs=outputs)
        return model

    input_shape = (max_spectrogram_length, n_mels)
    model = create_model(input_shape, vocab_size, units=256, max_length=max_length)

    # Compilar el modelo
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

print(model.summary())


In [ ]:
# ELIMINAR Paso 7.1: Cargar datos alineados desde Google Drive

aligned_mel_spectrograms = []
aligned_input_ids = []
languages_list = ['Italian', 'Spanish', 'Swahili']

for language_name in languages_list:
    aligned_data_path = os.path.join(base_dir, f'aligned_data_{language_name}.npz')
    if os.path.exists(aligned_data_path):
        print(f"\nCargando datos alineados completos para {language_name}...")
        data = np.load(aligned_data_path, allow_pickle=True)
        mel_spectrograms = data['mel_spectrograms']
        input_ids = data['input_ids']
        aligned_mel_spectrograms.extend(mel_spectrograms)
        aligned_input_ids.extend(input_ids)
        print(f"Datos cargados: {len(mel_spectrograms)} muestras para {language_name}")
    else:
        print(f"\nDatos alineados para {language_name} no encontrados.")

print(f"\nTotal de espectrogramas mel cargados: {len(aligned_mel_spectrograms)}")
print(f"Total de secuencias de entrada cargadas: {len(aligned_input_ids)}")

if len(aligned_mel_spectrograms) == len(aligned_input_ids):
    print("Todos los datos alineados han sido cargados exitosamente.")
else:
    print("Desajuste en el número de espectrogramas mel y secuencias de entrada.")


In [ ]:
# Celda 7.1: Verificar la disponibilidad de la GPU y monitorear recursos

import tensorflow as tf
import psutil
import time
from tqdm import tqdm

# Verificar la disponibilidad de la GPU
print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

# Función para monitorear recursos
def monitor_resources(duration=30):
    for _ in tqdm(range(duration), desc="Monitoreando recursos"):
        cpu = psutil.cpu_percent(interval=1)
        memory = psutil.virtual_memory().percent
        gpu_available = tf.config.list_physical_devices('GPU')
        if gpu_available:
            # Obtener memoria GPU usando TensorFlow
            try:
                gpu_details = tf.config.experimental.get_memory_info('GPU:0')
                gpu_memory = gpu_details['current'] / (1024 ** 3)  # GB
                print(f"CPU: {cpu}% | Memoria RAM: {memory}% | GPU Memoria: {gpu_memory:.2f} GB")
            except:
                print(f"CPU: {cpu}% | Memoria RAM: {memory}% | GPU: Disponible pero no se puede obtener información de memoria.")
        else:
            print(f"CPU: {cpu}% | Memoria RAM: {memory}% | GPU: No disponible")
        time.sleep(1)

# Ejecutar la monitorización por 30 segundos
monitor_resources(30)


In [ ]:
# Celda 8: Entrenar el modelo

# Callbacks para entrenamiento
checkpoint_cb = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "best_model.h5"),
    monitor="val_loss",
    verbose=1,
    save_best_only=True,
    mode="min"
)

early_stopping_cb = EarlyStopping(
    monitor="val_loss",
    patience=5,
    verbose=1,
    restore_best_weights=True
)

tensorboard_cb = TensorBoard(log_dir=logs_dir)

# Entrenar el modelo
epochs = 15
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
    callbacks=[checkpoint_cb, early_stopping_cb, tensorboard_cb]
)

print("Entrenamiento completado.")


In [ ]:
# Paso 8: Dividir datos en conjuntos de entrenamiento y validación

X_train, X_val, y_train, y_val = train_test_split(
    aligned_mel_spectrograms,
    aligned_input_ids,
    test_size=0.1,
    random_state=42
)

print(f"Datos de entrenamiento: {len(X_train)} muestras")
print(f"Datos de validación: {len(X_val)} muestras")


In [ ]:
# Paso 8.1: Calcular la longitud máxima de las secuencias basado en el 95º percentil de las secuencias de entrenamiento

train_seq_lengths = [len(seq) for seq in y_train]
percentil = 95
max_length_train = int(np.percentile(train_seq_lengths, percentil))
print(f"Longitud máxima de secuencias en entrenamiento (percentil {percentil}%): {max_length_train}")

max_length = max_length_train
print(f"Valor de max_length establecido a: {max_length}")


In [ ]:
# Paso 9: Preparar los datos y definir los datasets

from tensorflow.keras.preprocessing.sequence import pad_sequences
from scipy.interpolate import interp1d

n_mels = 128
max_spectrogram_length = 20  # Ajustar según tus datos

def pad_or_truncate_spectrogram(spectrogram, max_length):
    if spectrogram.shape[1] != n_mels:
        x = np.arange(spectrogram.shape[1])
        f = interp1d(x, spectrogram, axis=1, kind='linear')
        spectrogram = f(np.linspace(0, spectrogram.shape[1] - 1, n_mels))

    time_steps = spectrogram.shape[0]
    if time_steps < max_length:
        pad_width = max_length - time_steps
        padded_spectrogram = np.pad(spectrogram, ((0, pad_width), (0, 0)), mode='constant')
    else:
        padded_spectrogram = spectrogram[:max_length, :]
    return padded_spectrogram

X_train_padded = np.array([pad_or_truncate_spectrogram(s, max_spectrogram_length) for s in X_train])
X_val_padded = np.array([pad_or_truncate_spectrogram(s, max_spectrogram_length) for s in X_val])

y_train_padded = pad_sequences(y_train, maxlen=max_length, padding='post', truncating='post')
y_val_padded = pad_sequences(y_val, maxlen=max_length, padding='post', truncating='post')

batch_size = 32

train_dataset = tf.data.Dataset.from_tensor_slices((X_train_padded, y_train_padded))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_padded, y_val_padded))
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Paso 10: Verificar los datasets

for X_batch, y_batch in train_dataset.take(1):
    print("Dataset de entrenamiento OK.")
    print("Forma de X_batch:", X_batch.shape)
    print("Forma de y_batch:", y_batch.shape)

for X_batch, y_batch in val_dataset.take(1):
    print("Dataset de validación OK.")
    print("Forma de X_batch:", X_batch.shape)
    print("Forma de y_batch:", y_batch.shape)


In [ ]:
# Paso 11: Definir la arquitectura del modelo con Bahdanau Attention

all_tokens = np.concatenate(y_train_padded)
vocab_size = int(np.max(all_tokens)) + 1
print(f"Tamaño del vocabulario: {vocab_size}")

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

def create_model(input_shape, vocab_size, units=256, max_length=15):
    inputs = layers.Input(shape=input_shape)
    x = layers.Masking(mask_value=0.0)(inputs)
    x = layers.Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)
    encoder_outputs = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
    encoder_outputs = layers.Dropout(0.2)(encoder_outputs)

    attention_layer = BahdanauAttention(units)
    query = encoder_outputs[:, -1, :]
    context_vector, attention_weights = attention_layer(query, encoder_outputs)

    context_vector_repeated = layers.RepeatVector(max_length)(context_vector)
    decoder_input = layers.Concatenate(axis=-1)([encoder_outputs, context_vector_repeated])

    x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(decoder_input)
    x = layers.Dropout(0.2)(x)
    outputs = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

input_shape = (max_length, n_mels)
model = create_model(input_shape, vocab_size, units=256, max_length=max_length)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# Paso 12: Definir funciones auxiliares y callbacks

def tokens_to_text(sequences, index_word):
    texts = []
    for seq in sequences:
        seq = [idx for idx in seq if idx != 0]
        if len(seq) == 0:
            text = ""
        else:
            words = [index_word.get(token, '') for token in seq]
            text = ' '.join(words)
        texts.append(text)
    return texts

# Cargar el diccionario index_word si tienes el tokenizer (opcional)
# Aquí asumimos que ya tienes index_word; si no, adaptarlo según tu tokenización.

# Crear callback WER/CER
class WerCerCallback(tf.keras.callbacks.Callback):
    def __init__(self, validation_data, index_word):
        super(WerCerCallback, self).__init__()
        self.validation_data = validation_data
        self.index_word = index_word

    def on_epoch_end(self, epoch, logs=None):
        y_trues = []
        y_preds = []

        for X_batch, y_batch in self.validation_data:
            y_pred = self.model.predict(X_batch, verbose=0)
            y_pred = tf.argmax(y_pred, axis=-1).numpy()
            y_true = y_batch.numpy()

            true_texts = tokens_to_text(y_true, self.index_word)
            pred_texts = tokens_to_text(y_pred, self.index_word)

            y_trues.extend(true_texts)
            y_preds.extend(pred_texts)

        wer_score = wer(y_trues, y_preds)
        cer_score = cer(y_trues, y_preds)
        print(f'\nEpoch {epoch+1} - WER: {wer_score:.4f}, CER: {cer_score:.4f}')

wer_cer_callback = WerCerCallback(validation_data=val_dataset, index_word=index_word)

# Otros callbacks
checkpoint = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, 'model_epoch_{epoch:02d}_val_loss_{val_loss:.4f}.h5'),
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    mode='min'
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=1,
    restore_best_weights=True
)

tensorboard_callback = TensorBoard(
    log_dir=logs_dir
)


In [ ]:
# Paso 13: Ajustar el optimizador u otros parámetros (opcional)
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy'
)


In [ ]:
# Paso 14: Entrenar el modelo
epochs = 15
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
    callbacks=[checkpoint, early_stopping, tensorboard_callback, wer_cer_callback]
)


In [ ]:
# Paso 15: Guardar el modelo final
model.save(model_save_path)
print(f"Modelo guardado en {model_save_path}")


# TENSORFLOW LITE

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.3 MB/s eta 0:00:00


In [ ]:
# Celda 1: Imports y montaje de Google Drive

import os
import time
import numpy as np
import tensorflow as tf
import jiwer
from jiwer import wer, cer
from tensorflow.lite.python.interpreter import Interpreter
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Celda 2: Definir rutas reales

# Modelo Keras entrenado
checkpoint_dir   = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/checkpoints"
keras_model_file = os.path.join(checkpoint_dir, "best_model_multiling_subwords.h5")

# Carpeta de salida de TFLite
tflite_dir = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/tflite_models"
os.makedirs(tflite_dir, exist_ok=True)

# Datos de prueba (NPZ) para calibración y evaluación
aligned_dir = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/aligned_data"
test_npz    = os.path.join(aligned_dir, "aligned_data_Italian.npz")  # Ajusta idioma aquí


In [ ]:
# Celda 3: Función de conversión y generación de .tflite

def convert_and_save(converter, name):
    tflite_path = os.path.join(tflite_dir, f"{name}.tflite")
    tflite_model = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    size_mb = os.path.getsize(tflite_path) / (1024**2)
    print(f"✅ {name}: {size_mb:.2f} MB -> {tflite_path}")
    return tflite_path

# Load the Keras model and create the converter
model = tf.keras.models.load_model(keras_model_file, compile=False)  # Load the Keras model first
converter = tf.lite.TFLiteConverter.from_keras_model(model)  # Create the converter from the model

tflite_paths = []
tflite_paths.append(convert_and_save(converter, "float32"))

# Dynamic-range quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_paths.append(convert_and_save(converter, "dr_quant"))

# Float16 quantization
converter.target_spec.supported_types = [tf.float16]
tflite_paths.append(convert_and_save(converter, "float16_quant"))

# Integer quantization completo
def representative_dataset():
    data = np.load(test_npz)
    for mel in data["mel_spectrograms"][:100]:
        yield [mel.astype(np.float32)]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8
tflite_paths.append(convert_and_save(converter, "int8_quant"))

TypeError: Could not locate class 'LSTM'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'class_name': 'LSTM', 'config': {'name': 'lstm_2', 'trainable': True, 'dtype': 'float32', 'return_sequences': True, 'return_state': False, 'go_backwards': False, 'stateful': False, 'unroll': False, 'time_major': False, 'units': 256, 'activation': 'tanh', 'recurrent_activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'class_name': 'GlorotUniform', 'config': {'seed': None}, 'shared_object_id': 6}, 'recurrent_initializer': {'class_name': 'Orthogonal', 'config': {'gain': 1.0, 'seed': None}, 'shared_object_id': 7}, 'bias_initializer': {'class_name': 'Zeros', 'config': {}, 'shared_object_id': 8}, 'unit_forget_bias': True, 'kernel_regularizer': None, 'recurrent_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'recurrent_constraint': None, 'bias_constraint': None, 'dropout': 0

In [ ]:
# Celda 4: Evaluación de inferencia y cálculo de WER/CER

# Carga de datos de prueba
npz     = np.load(test_npz, allow_pickle=True)
mels    = npz["mel_spectrograms"]
ids_gt  = npz["input_ids"]

# Evaluación del baseline Keras
from tensorflow.keras.models import load_model
model      = load_model(keras_model_file, compile=False)
decoded_tf = []
for mel in tqdm(mels[:100], desc="Baseline infer"):
    pred = model.predict(mel[None,...])
    decoded_tf.append(pred.argmax(-1).flatten().tolist())

print("\n— Baseline Keras —")
print("WER:", wer(ids_gt[:100], decoded_tf))
print("CER:", cer(ids_gt[:100], decoded_tf))

# Función auxiliar para TFLite
def run_tflite(path):
    interp = Interpreter(model_path=path)
    interp.allocate_tensors()
    inp_idx, out_idx = interp.get_input_details()[0]["index"], interp.get_output_details()[0]["index"]
    preds, times = [], []
    for mel in tqdm(mels[:100], desc=f"Infer {os.path.basename(path)}"):
        interp.set_tensor(inp_idx, mel.astype(np.float32)[None,...])
        start = time.time()
        interp.invoke()
        times.append(time.time() - start)
        out = interp.get_tensor(out_idx)
        preds.append(out.argmax(-1).flatten().tolist())
    return preds, np.mean(times)*1000

for path in tflite_paths:
    preds, latency = run_tflite(path)
    print(f"\n— {os.path.basename(path)} —")
    print(f"Latency: {latency:.1f} ms/sample")
    print("WER:", wer(ids_gt[:100], preds))
    print("CER:", cer(ids_gt[:100], preds))



In [ ]:
# Celda 5: Tabla resumen de métricas (WER, CER, latencia, tamaño)

import pandas as pd

metrics = []

# Baseline
metrics.append({
    'model': 'baseline',
    'size_MB': None,
    'latency_ms': None,
    'WER': wer(ids_gt[:100], decoded_tf),
    'CER': cer(ids_gt[:100], decoded_tf)
})

# TFLite
for path in tflite_paths:
    name = os.path.splitext(os.path.basename(path))[0]
    size = os.path.getsize(path) / (1024**2)
    preds, latency = run_tflite(path)
    metrics.append({
        'model': name,
        'size_MB': size,
        'latency_ms': latency,
        'WER': wer(ids_gt[:100], preds),
        'CER': cer(ids_gt[:100], preds)
    })

df_metrics = pd.DataFrame(metrics)
print(df_metrics)



# **Testing si funciona**

In [ ]:
# Actualizar gradio, transformers, torchaudio y safetensors
!pip install --upgrade gradio transformers torchaudio safetensors


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Definir la ruta base en Google Drive
google_drive_base = "/content/drive/My Drive/TinyML for Translation/SST & TTS"

# Ruta donde se guardó el modelo entrenado
trained_model_path = os.path.join(google_drive_base, "Multilingual", "exported_model", "epoch_10")

# Verificar que la ruta existe
if not os.path.exists(trained_model_path):
    raise FileNotFoundError(f"La ruta {trained_model_path} no existe. Verifica el path correcto.")
else:
    print(f"La ruta {trained_model_path} existe. Contenido:")
    for file in os.listdir(trained_model_path):
        print(f"- {file}")



Mounted at /content/drive
La ruta /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/epoch_10 existe. Contenido:
- config.json
- generation_config.json
- model.safetensors
- preprocessor_config.json
- tokenizer_config.json
- special_tokens_map.json
- added_tokens.json
- vocab.json
- merges.txt
- normalizer.json


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Cargar el procesador y el modelo desde la ruta especificada
processor = WhisperProcessor.from_pretrained(trained_model_path)
model = WhisperForConditionalGeneration.from_pretrained(trained_model_path)

# Mover el modelo al dispositivo adecuado
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Modelo y procesador cargados exitosamente.")


Modelo y procesador cargados exitosamente.


In [ ]:
import torchaudio
import numpy as np

def transcribe(audio_file):
    # Cargar el audio utilizando torchaudio
    waveform, sample_rate = torchaudio.load(audio_file)

    # Si el audio tiene más de un canal, convertirlo a mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Re-samplear si la tasa de muestreo no es 16000
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    # Convertir a numpy y procesar el audio con el procesador
    inputs = processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    # Generar la transcripción
    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    # Decodificar la transcripción
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription


In [ ]:
/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/Spanish/wavprocessed/common_voice_es_20255301.wav


In [ ]:

audio_test_path = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/wavpreprocessed/common_voice_es_20255301.wav"

if not os.path.exists(audio_test_path):
    raise FileNotFoundError(f"El archivo de audio {audio_test_path} no existe.")
else:
    print(f"El archivo de audio {audio_test_path} está disponible.")



El archivo de audio /content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/wavpreprocessed/common_voice_es_20255301.wav está disponible.


In [ ]:
import gradio as gr

# Definir la interfaz de Gradio sin el parámetro 'source'
iface = gr.Interface(
    fn=transcribe,  # La función que definimos anteriormente
    inputs=gr.Audio(type="filepath"),  # Entrada de audio mediante subida de archivo
    outputs="text",  # Salida como texto
    title="Transcripción de Voz Multilingüe",
    description="Sube un archivo de audio y el modelo Whisper lo transcribirá.",
    examples=[
        [audio_test_path],
        # Puedes añadir más ejemplos si lo deseas
    ],
    allow_flagging="never"  # Opcional: deshabilitar la bandera para reportar problemas
)

# Iniciar la interfaz
iface.launch(debug=True)


NameError: name 'transcribe' is not defined

# **PYTORCH**

In [ ]:
# Paso 2: Instalar y actualizar bibliotecas
!pip install --upgrade transformers safetensors gradio torchaudio jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 

In [ ]:
# Paso 1: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



# Paso 3: Importar librerías
import os
import warnings
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration, GenerationConfig
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from jiwer import wer, cer
from torch.cuda.amp import autocast, GradScaler
import gc
import gradio as gr

# Paso 4: Suprimir warnings
warnings.filterwarnings("ignore")

# Paso 5: Definir rutas de datos
audio_dir = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/wavpreprocessed",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/wavpreprocessed",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/wavpreprocessed"
}

pt_dir = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/preprocessed_transcriptionsit.pt",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/preprocessed_transcriptionses.pt",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/preprocessed_transcriptionssw.pt"
}

# Paso 6: Inicializar el procesador
print("Inicializando el procesador...")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# Paso 7: Añadir tokens de idioma al tokenizador
language_tokens = {"spanish": "<|es|>", "italian": "<|it|>", "swahili": "<|sw|>"}
processor.tokenizer.add_tokens(list(language_tokens.values()))

# Paso 8: Inicializar el modelo y ajustar el tamaño de los embeddings
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model.resize_token_embeddings(len(processor.tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Paso 9: Definir funciones para actualizar y cargar datos
def update_preprocessed_data(language):
    with open(pt_dir[language], 'rb') as f:
        data = torch.load(f)
    filtered_data = []
    audio_directory = audio_dir[language]
    for item in data:
        audio_file = os.path.join(audio_directory, item["audio_file"])
        if os.path.isfile(audio_file):
            filtered_data.append(item)
    torch.save(filtered_data, pt_dir[language])
    print(f"Archivo .pt actualizado para {language}: {len(filtered_data)} muestras.")
    return filtered_data

def load_and_prepare_multilingual_data(languages):
    preprocessed_data = []
    for language in languages:
        data = update_preprocessed_data(language)
        for item in data:
            item['language'] = language
            item['language_code'] = language_tokens[language].strip('<|>')
            preprocessed_data.append(item)
    return preprocessed_data

# Definir el Dataset personalizado
class AudioTextDataset(Dataset):
    def __init__(self, audio_dirs, preprocessed_data):
        self.audio_dirs = audio_dirs
        self.preprocessed_data = preprocessed_data

    def __len__(self):
        return len(self.preprocessed_data)

    def __getitem__(self, idx):
        item = self.preprocessed_data[idx]
        language = item["language"]
        audio_file = os.path.join(self.audio_dirs[language], item["audio_file"])

        try:
            waveform, sample_rate = torchaudio.load(audio_file)
            if sample_rate != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
                waveform = resampler(waveform)
            input_features = processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt").input_features

            # Añadir el token de idioma al inicio de las etiquetas
            language_token = language_tokens[language]
            language_token_id = processor.tokenizer.convert_tokens_to_ids(language_token)
            labels = torch.cat([torch.tensor([language_token_id]), item["labels"]])

            return {
                "input_features": input_features.squeeze(0),
                "labels": labels
            }
        except Exception as e:
            print(f"Error al cargar el archivo de audio {audio_file}: {e}")
            return None

# Definir la función collate_fn
def collate_fn(batch):
    # Filtrar None
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    input_features = [item["input_features"] for item in batch]
    labels = [item["labels"] for item in batch]

    # Padding de input_features
    input_features_padded = torch.nn.utils.rnn.pad_sequence(input_features, batch_first=True, padding_value=0.0)

    # Padding de labels
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=processor.tokenizer.pad_token_id)

    return {
        "input_features": input_features_padded,
        "labels": labels_padded
    }

# Definir la función de entrenamiento
def train_multilingual(model, dataloader, epochs=4, lr=1e-4, gradient_accumulation_steps=4):
    optimizer = AdamW(model.parameters(), lr=lr)
    scaler = GradScaler()
    writer = SummaryWriter(log_dir='/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/tensorboard_logs/')
    global_step = 0
    save_interval = 1000  # Ajusta según tus necesidades

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        model.train()
        epoch_loss = 0
        for step, batch in enumerate(tqdm(dataloader)):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            # Verificar el tamaño del batch
            batch_size_current = input_features.size(0)
            if batch_size_current != 16:
                print(f"¡Advertencia! Tamaño del batch en el paso {step}: {batch_size_current}")

            with autocast():
                outputs = model(input_features=input_features, labels=labels)
                loss = outputs.loss / gradient_accumulation_steps  # Normalizar pérdida

            scaler.scale(loss).backward()

            if (step + 1) % gradient_accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                torch.cuda.empty_cache()  # Limpiar caché de la GPU

            epoch_loss += loss.item() * gradient_accumulation_steps
            writer.add_scalar("Loss/train", loss.item() * gradient_accumulation_steps, global_step)
            global_step += 1

            if global_step % save_interval == 0:
                checkpoint_path = f'/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/checkpoint_step_{global_step}'
                model.save_pretrained(checkpoint_path, save_safetensors=True)
                processor.save_pretrained(checkpoint_path)
                print(f"Modelo y procesador guardados en {checkpoint_path}")

        avg_loss = epoch_loss / len(dataloader)
        print(f"Loss promedio para Epoch {epoch+1}: {avg_loss:.4f}")
        writer.add_scalar("Loss/epoch", avg_loss, epoch)

        epoch_model_path = f'/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/epoch_{epoch+1}'
        model.save_pretrained(epoch_model_path, save_safetensors=True)
        processor.save_pretrained(epoch_model_path)
        print(f"Modelo y procesador guardados en {epoch_model_path}")

    writer.close()

# Paso 10: Definir la función de evaluación
def evaluate_multilingual(model, dataloader):
    model.eval()
    references = []
    hypotheses = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            generated_ids = model.generate(input_features)
            transcriptions = processor.batch_decode(generated_ids, skip_special_tokens=True)
            references_batch = processor.batch_decode(labels, skip_special_tokens=True)

            # Remover el token de idioma de las referencias
            references_batch = [ref.split(' ', 1)[1] if len(ref.split(' ', 1)) > 1 else ref for ref in references_batch]
            hypotheses_batch = [hyp.split(' ', 1)[1] if len(hyp.split(' ', 1)) > 1 else hyp for hyp in transcriptions]

            references.extend(references_batch)
            hypotheses.extend(hypotheses_batch)

    total_wer = wer(references, hypotheses)
    total_cer = cer(references, hypotheses)

    print(f"WER: {total_wer:.4f}, CER: {total_cer:.4f}")
    return total_wer, total_cer

# Paso 11: Cargar y preparar los datos multilingües
languages = ["italian", "spanish", "swahili"]
preprocessed_data = load_and_prepare_multilingual_data(languages)
dataset = AudioTextDataset(audio_dir, preprocessed_data)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn, num_workers=0)

# Paso 12: Entrenar el modelo multilingüe
train_multilingual(model, dataloader, epochs=3, lr=1e-4, gradient_accumulation_steps=4)

# Paso 13: Evaluar el modelo multilingüe
evaluate_multilingual(model, dataloader)


Mounted at /content/drive
Inicializando el procesador...
Archivo .pt actualizado para italian: 4000 muestras.
Archivo .pt actualizado para spanish: 4000 muestras.
Archivo .pt actualizado para swahili: 4000 muestras.
Epoch 1/3


  0%|          | 1/1500 [00:12<5:14:57, 12.61s/it]Exception in thread Thread-19:
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.10/dist-packages/tensorboard/summary/writer/event_file_writer.py", line 244, in run
    self._run()
  File "/usr/local/lib/python3.10/dist-packages/tensorboard/summary/writer/event_file_writer.py", line 289, in _run
    self._record_writer.flush()
  File "/usr/local/lib/python3.10/dist-packages/tensorboard/summary/writer/record_writer.py", line 43, in flush
    self._writer.flush()
  File "/usr/local/lib/python3.10/dist-packages/tensorflow/python/lib/io/file_io.py", line 221, in flush
    self._writable_file.flush()
tensorflow.python.framework.errors_impl.FailedPreconditionError: /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/tensorboard_logs/events.out.tfevents.1727895831.8c20fee41922.8972.2; Transport endpoint is not connected
 6

Modelo y procesador guardados en /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/checkpoint_step_1000


100%|██████████| 1500/1500 [5:13:32<00:00, 12.54s/it]


Loss promedio para Epoch 1: 0.1688
Modelo y procesador guardados en /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/epoch_1
Epoch 2/3


...

 67%|██████▋   | 1000/1500 [2:23:07<1:12:54,  8.75s/it]

Modelo y procesador guardados en /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/checkpoint_step_4000


100%|██████████| 1500/1500 [3:36:03<00:00,  8.64s/it]


Loss promedio para Epoch 3: 0.0389
Modelo y procesador guardados en /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/epoch_3


  0%|          | 0/1500 [00:00<?, ?it/s]Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
 94%|█████████▍| 1415/1500 [2:14:25<07:54,  5.58s/it]

In [ ]:
# Paso 1: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Paso 2: Instalar y actualizar bibliotecas
!pip install --upgrade transformers safetensors gradio torchaudio jiwer

# Paso 3: Importar librerías
import os
import warnings
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration, GenerationConfig
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from jiwer import wer, cer
from torch.cuda.amp import autocast, GradScaler
import gc
import gradio as gr

# Paso 4: Suprimir warnings
warnings.filterwarnings("ignore")

# Paso 5: Definir rutas de datos
google_drive_base = "/content/drive/My Drive/TinyML for Translation/SST & TTS/"
local_base = "/content/SST_TTS/"

# Definir directorios de audio y preprocesados
audio_dir = {
    "italian": os.path.join(local_base, "Italian/wavpreprocessed"),
    "spanish": os.path.join(local_base, "Spanish/wavpreprocessed"),
    "swahili": os.path.join(local_base, "Swahili/wavpreprocessed")
}

preprocessed_features_dir = {
    "italian": os.path.join(local_base, "Italian/preprocessed_features"),
    "spanish": os.path.join(local_base, "Spanish/preprocessed_features"),
    "swahili": os.path.join(local_base, "Swahili/preprocessed_features")
}

# Definir rutas de archivos de transcripciones
pt_dir = {
    "italian": os.path.join(local_base, "Italian/preprocessed_transcriptionsit.pt"),
    "spanish": os.path.join(local_base, "Spanish/preprocessed_transcriptionses.pt"),
    "swahili": os.path.join(local_base, "Swahili/preprocessed_transcriptionssw.pt")
}

# Crear directorios locales completos si no existen (solo directorios, no archivos)
for dir_path in list(audio_dir.values()) + list(preprocessed_features_dir.values()):
    os.makedirs(dir_path, exist_ok=True)  # Asegura que todo el directorio exista

# Definir códigos de idioma
language_codes = {"italian": "it", "spanish": "es", "swahili": "sw"}

# Paso 6: Inicializar el procesador
print("Inicializando el procesador...")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# Paso 7: Añadir tokens de idioma al tokenizador
language_tokens = {"spanish": "<|es|>", "italian": "<|it|>", "swahili": "<|sw|>"}
processor.tokenizer.add_tokens(list(language_tokens.values()))
print("Tokens de idioma añadidos al tokenizador.")

# Paso 8: Inicializar el modelo y ajustar el tamaño de los embeddings
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model.resize_token_embeddings(len(processor.tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Modelo cargado en {device}.")

# Paso 9: Definir funciones para copiar y cargar datos
def copiar_datos_a_local():
    for language in ["italian", "spanish", "swahili"]:
        drive_audio_path = os.path.join(google_drive_base, language.capitalize(), "wavpreprocessed")
        local_audio_path = audio_dir[language]
        if not os.path.exists(local_audio_path) or len(os.listdir(local_audio_path)) == 0:
            print(f"Copiando audios para {language}...")
            !cp -r "{drive_audio_path}/." "{local_audio_path}/"
        else:
            print(f"Audios para {language} ya están copiados.")

        drive_pt_path = os.path.join(google_drive_base, language.capitalize(), f"preprocessed_transcriptions{language_codes[language]}.pt")
        local_pt_path = pt_dir[language]
        if not os.path.exists(local_pt_path):
            print(f"Copiando transcripciones para {language}...")
            !cp "{drive_pt_path}" "{local_pt_path}"
        else:
            print(f"Transcripciones para {language} ya están copiadas.")

def update_preprocessed_data(language):
    try:
        with open(pt_dir[language], 'rb') as f:
            data = torch.load(f)
    except FileNotFoundError:
        print(f"Archivo {pt_dir[language]} no encontrado para {language}.")
        return []
    filtered_data = []
    for item in data:
        audio_file = item["audio_file"]
        audio_path = os.path.join(audio_dir[language], audio_file)
        if os.path.isfile(audio_path):
            filtered_data.append(item)
    torch.save(filtered_data, pt_dir[language])
    print(f"Archivo .pt actualizado para {language}: {len(filtered_data)} muestras.")
    return filtered_data

def load_and_prepare_multilingual_data(languages):
    preprocessed_data = []
    for language in languages:
        data = update_preprocessed_data(language)
        for item in data:
            item['language'] = language
            item['language_code'] = language_tokens[language].strip('<|>')
            preprocessed_data.append(item)
    return preprocessed_data

# Ejecutar la función para copiar datos
copiar_datos_a_local()

# Paso 10: Preprocesar y guardar input_features
def preprocess_and_save_features(language):
    print(f"Preprocesando y guardando input_features para {language}...")
    audio_files = os.listdir(audio_dir[language])
    for audio_file in tqdm(audio_files, desc=f"Preprocesando {language}"):
        audio_path = os.path.join(audio_dir[language], audio_file)
        feature_path = os.path.join(preprocessed_features_dir[language], os.path.splitext(audio_file)[0] + ".pt")
        if not os.path.isfile(feature_path):
            try:
                waveform, sample_rate = torchaudio.load(audio_path)
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)  # Mezclar canales si es estéreo
                if sample_rate != 16000:
                    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
                    waveform = resampler(waveform)
                input_features = processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt").input_features.squeeze(0)
                torch.save(input_features, feature_path)
            except Exception as e:
                print(f"Error al preprocesar {audio_file}: {e}")
        else:
            # Puedes comentar la siguiente línea si no quieres ver mensajes sobre archivos ya preprocesados
            # print(f"input_features para {audio_file} ya existen.")
            pass

# Ejecutar preprocesamiento
languages = ["italian", "spanish", "swahili"]
for language in languages:
    preprocess_and_save_features(language)

# Paso 11: Definir el Dataset personalizado
class AudioTextDataset(Dataset):
    def __init__(self, preprocessed_features_dirs, preprocessed_data):
        self.preprocessed_features_dirs = preprocessed_features_dirs
        self.preprocessed_data = preprocessed_data

    def __len__(self):
        return len(self.preprocessed_data)

    def __getitem__(self, idx):
        item = self.preprocessed_data[idx]
        language = item["language"]
        audio_file = item["audio_file"]
        feature_path = os.path.join(self.preprocessed_features_dirs[language], os.path.splitext(audio_file)[0] + ".pt")

        try:
            # Cargar los input_features preprocesados
            input_features = torch.load(feature_path)

            # Añadir el token de idioma al inicio de las etiquetas
            language_token = language_tokens[language]
            language_token_id = processor.tokenizer.convert_tokens_to_ids(language_token)
            labels = torch.cat([torch.tensor([language_token_id]), item["labels"]])

            return {
                "input_features": input_features,
                "labels": labels
            }
        except Exception as e:
            print(f"Error al cargar los input_features para {audio_file}: {e}")
            return None

# Paso 12: Definir la función collate_fn
def collate_fn(batch):
    # Filtrar None
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    input_features = [item["input_features"] for item in batch]
    labels = [item["labels"] for item in batch]

    # Padding de input_features
    input_features_padded = torch.nn.utils.rnn.pad_sequence(input_features, batch_first=True, padding_value=0.0)

    # Padding de labels
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=processor.tokenizer.pad_token_id)

    return {
        "input_features": input_features_padded,
        "labels": labels_padded
    }

# Paso 13: Cargar y preparar los datos multilingües
preprocessed_data = load_and_prepare_multilingual_data(languages)
dataset = AudioTextDataset(preprocessed_features_dir, preprocessed_data)

# Definir el DataLoader con múltiples workers y pin_memory
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,  # Puedes ajustar este número según el entorno
    pin_memory=True  # Mejora la transferencia a la GPU
)

# Paso 14: Definir la función de entrenamiento
def train_multilingual(model, dataloader, epochs=3, lr=1e-4, gradient_accumulation_steps=4):
    optimizer = AdamW(model.parameters(), lr=lr)
    scaler = GradScaler()
    writer = SummaryWriter(log_dir='/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/tensorboard_logs/')
    global_step = 0
    save_interval = 5000  # Ajustar según sea necesario

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        model.train()
        epoch_loss = 0
        for step, batch in enumerate(tqdm(dataloader, desc=f"Entrenando Epoch {epoch+1}")):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            # Verificar el tamaño del batch
            batch_size_current = input_features.size(0)
            if batch_size_current != 16:
                print(f"¡Advertencia! Tamaño del batch en el paso {step}: {batch_size_current}")

            with autocast():
                outputs = model(input_features=input_features, labels=labels)
                loss = outputs.loss / gradient_accumulation_steps  # Normalizar pérdida

            scaler.scale(loss).backward()

            if (step + 1) % gradient_accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                torch.cuda.empty_cache()  # Limpiar caché de la GPU

            epoch_loss += loss.item() * gradient_accumulation_steps
            writer.add_scalar("Loss/train", loss.item() * gradient_accumulation_steps, global_step)
            global_step += 1

            if global_step % save_interval == 0:
                checkpoint_path = f'/content/drive/My Drive/TinyML for Translation/SST & TTS/exported_model/checkpoint_step_{global_step}'
                os.makedirs(checkpoint_path, exist_ok=True)
                model.save_pretrained(checkpoint_path, save_safetensors=True)
                processor.save_pretrained(checkpoint_path)
                print(f"Modelo y procesador guardados en {checkpoint_path}")

        avg_loss = epoch_loss / len(dataloader)
        print(f"Loss promedio para Epoch {epoch+1}: {avg_loss:.4f}")
        writer.add_scalar("Loss/epoch", avg_loss, epoch)

        epoch_model_path = f'/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/epoch_{epoch+1}'
        os.makedirs(epoch_model_path, exist_ok=True)
        model.save_pretrained(epoch_model_path, save_safetensors=True)
        processor.save_pretrained(epoch_model_path)
        print(f"Modelo y procesador guardados en {epoch_model_path}")

        # Liberar memoria después de cada epoch
        gc.collect()
        torch.cuda.empty_cache()

    writer.close()

# Paso 15: Definir la función de evaluación
def evaluate_multilingual(model, dataloader):
    model.eval()
    references = []
    hypotheses = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluando"):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            # Crear attention_mask
            attention_mask = (input_features != processor.tokenizer.pad_token_id).long()

            # Generar la transcripción
            generated_ids = model.generate(
                input_features=input_features,
                attention_mask=attention_mask
            )
            transcriptions = processor.batch_decode(generated_ids, skip_special_tokens=True)
            references_batch = processor.batch_decode(labels, skip_special_tokens=True)

            # Remover el token de idioma de las referencias
            references_batch = [ref.split(' ', 1)[1] if len(ref.split(' ', 1)) > 1 else ref for ref in references_batch]
            hypotheses_batch = [hyp.split(' ', 1)[1] if len(hyp.split(' ', 1)) > 1 else hyp for hyp in transcriptions]

            references.extend(references_batch)
            hypotheses.extend(hypotheses_batch)

    total_wer = wer(references, hypotheses)
    total_cer = cer(references, hypotheses)

    print(f"WER: {total_wer:.4f}, CER: {total_cer:.4f}")
    return total_wer, total_cer


# Ejecutar el entrenamiento
train_multilingual(model, dataloader, epochs=10, lr=1e-4, gradient_accumulation_steps=4)

# Evaluar el modelo entrenado
evaluate_multilingual(model, dataloader)


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

Modelo y procesador guardados en /content/drive/My Drive/TinyML for Translation/SST & TTS/exported_model/checkpoint_step_15000
Loss promedio para Epoch 10: 0.0074


Modelo y procesador guardados en /content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/epoch_10


Evaluando: 100%|██████████| 1500/1500 [1:47:04<00:00,  4.28s/it]


WER: 0.5906, CER: 0.4967


(0.5906285274040098, 0.4967391257771893)

Actualizar el .pt con las transcripciones foneticas

In [ ]:
import pandas as pd
from transformers import WhisperProcessor
import torch
from tqdm import tqdm
import os
# Paso 1: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
# Inicializar el procesador de Whisper
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

# Definir ruta al archivo JSON de transcripciones para Italiano
json_file = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/transcriptionses.json"

# Cargar las transcripciones desde el archivo JSON
transcriptions = pd.read_json(json_file)

# Añadir el token de idioma para el italiano
language_token = "[LANG_ES]"

# Función para preprocesar los datos de las transcripciones fonéticas con padding
def preprocess_transcriptions(transcriptions, max_length=256):
    preprocessed_data = []
    for _, row in tqdm(transcriptions.iterrows(), total=transcriptions.shape[0], desc="Preprocesando transcripciones"):
        # Tokenizar las transcripciones fonéticas
        labels = processor.tokenizer(language_token + " " + row["phonetics"], return_tensors="pt", padding="max_length", max_length=max_length, truncation=True).input_ids.squeeze(0)
        attention_mask = processor.tokenizer(row["phonetics"], return_tensors="pt", padding="max_length", max_length=max_length, truncation=True).attention_mask.squeeze(0)

        # Almacenar los resultados, manteniendo la referencia del archivo de audio
        preprocessed_data.append({
            "audio_file": row["audio_file"],
            "labels": labels,  # Etiquetas tokenizadas con padding
            "attention_mask": attention_mask  # Máscara de atención con padding
        })

    return preprocessed_data

# Preprocesar las transcripciones con padding (ajustar max_length según tus datos)
preprocessed_data = preprocess_transcriptions(transcriptions, max_length=256)

# Definir la ruta de salida para los datos preprocesados
output_file = "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/preprocessed_transcriptionses.pt"

# Guardar los datos preprocesados usando torch.save (para mantener compatibilidad con PyTorch)
torch.save(preprocessed_data, output_file)

print(f"Datos preprocesados guardados en {output_file}")


Mounted at /content/drive


Preprocesando transcripciones: 100%|██████████| 14730/14730 [00:36<00:00, 401.93it/s]


Datos preprocesados guardados en /content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/preprocessed_transcriptionses.pt


# Using Whisper



---



---



---



---



In [ ]:
import os
import warnings  # Para suprimir warnings
from google.colab import drive
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from jiwer import wer, cer
from torch.cuda.amp import autocast, GradScaler

# Suprimir warnings
warnings.filterwarnings("ignore")

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas en Google Drive o local para cada idioma
audio_dir = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/wavpreprocessed",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/wavpreprocessed",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/wavpreprocessed"
}

pt_dir = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/preprocessed_transcriptionsit.pt",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/preprocessed_transcriptionses.pt",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/preprocessed_transcriptionssw.pt"
}

# Inicializar el procesador
print("Inicializando el procesador...")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# Verificar si CUDA está disponible y configurar el dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Función para sincronizar y actualizar los datos preprocesados
def update_preprocessed_data(language):
    with open(pt_dir[language], 'rb') as f:
        data = torch.load(f)

    # Filtrar entradas con archivos de audio existentes
    filtered_data = []
    audio_directory = audio_dir[language]
    for item in data:
        audio_file = os.path.join(audio_directory, item["audio_file"])
        if os.path.isfile(audio_file):
            filtered_data.append(item)

    # Guardar el archivo .pt actualizado
    updated_pt_path = pt_dir[language]
    torch.save(filtered_data, updated_pt_path)
    print(f"Archivo .pt actualizado para {language}: {len(filtered_data)} muestras.")

    return filtered_data

# Dataset personalizado
class AudioTextDataset(Dataset):
    def __init__(self, audio_dir, preprocessed_data):
        self.audio_dir = audio_dir
        self.preprocessed_data = preprocessed_data

    def __len__(self):
        return len(self.preprocessed_data)

    def __getitem__(self, idx):
        item = self.preprocessed_data[idx]
        audio_file = os.path.join(self.audio_dir, item["audio_file"])

        try:
            # Cargar el archivo de audio
            waveform, sample_rate = torchaudio.load(audio_file)
            # Asegurar que la tasa de muestreo sea de 16000 Hz
            if sample_rate != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
                waveform = resampler(waveform)
            input_features = processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt").input_features
            return {
                "input_features": input_features.squeeze(0),
                "labels": item["labels"]
            }
        except Exception as e:
            # Registrar el error y omitir la muestra
            print(f"Error al cargar el archivo de audio {audio_file}: {e}")
            return None

# Función collate_fn modificada
def collate_fn(batch):
    # Filtrar None
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    input_features = [item["input_features"] for item in batch]
    labels = [item["labels"] for item in batch]

    # Padding de input_features
    input_features_padded = torch.nn.utils.rnn.pad_sequence(input_features, batch_first=True, padding_value=0.0)

    # Padding de labels
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=processor.tokenizer.pad_token_id)

    return {
        "input_features": input_features_padded,
        "labels": labels_padded
    }

# Función de entrenamiento por idioma
def train(model, dataloader, language, epochs=5, lr=1e-4):
    optimizer = AdamW(model.parameters(), lr=lr)
    scaler = GradScaler()
    writer = SummaryWriter(log_dir=f'/content/drive/My Drive/TinyML for Translation/SST & TTS/{language}/tensorboard_logs/')

    global_step = 0
    save_interval = 1000  # Ajusta según tus necesidades

    for epoch in range(epochs):
        print(f"Entrenando {language}, Epoch {epoch+1}/{epochs}")
        model.train()
        epoch_loss = 0
        for step, batch in enumerate(tqdm(dataloader)):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            with autocast():
                outputs = model(input_features=input_features, labels=labels)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            writer.add_scalar(f"Loss/{language}/train", loss.item(), global_step)
            global_step += 1

            # Guardar checkpoint periódicamente
            if global_step % save_interval == 0:
                model.save_pretrained(f'/content/drive/My Drive/TinyML for Translation/SST & TTS/{language}/exported_model/checkpoint_step_{global_step}')

        avg_loss = epoch_loss / len(dataloader)
        print(f"Loss promedio para {language}, Epoch {epoch+1}: {avg_loss:.4f}")
        writer.add_scalar(f"Loss/{language}/epoch", avg_loss, epoch)

        # Guardar el modelo después de cada epoch
        model.save_pretrained(f'/content/drive/My Drive/TinyML for Translation/SST & TTS/{language}/exported_model/epoch_{epoch+1}')

    writer.close()

# Función de evaluación por idioma
def evaluate(model, dataloader, language):
    model.eval()
    references = []
    hypotheses = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            generated_ids = model.generate(input_features)
            transcriptions = processor.batch_decode(generated_ids, skip_special_tokens=True)
            references_batch = processor.batch_decode(labels, skip_special_tokens=True)

            references.extend(references_batch)
            hypotheses.extend(transcriptions)

    # Calcular WER y CER
    total_wer = wer(references, hypotheses)
    total_cer = cer(references, hypotheses)

    print(f"WER para {language}: {total_wer:.4f}, CER para {language}: {total_cer:.4f}")
    return total_wer, total_cer

# Función para entrenar y evaluar un idioma específico
def train_and_evaluate_language(language):
    print(f"Iniciando entrenamiento para {language}...")

    # Actualizar y cargar datos preprocesados
    preprocessed_data = update_preprocessed_data(language)
    if len(preprocessed_data) == 0:
        print(f"No hay datos válidos para {language}.")
        return
    dataset = AudioTextDataset(audio_dir[language], preprocessed_data)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

    # Inicializar el modelo para cada idioma
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
    model.to(device)

    # Entrenar y evaluar
    train(model, dataloader, language)
    evaluate(model, dataloader, language)

# Lista de idiomas a entrenar
languages = ["italian", "spanish", "swahili"]

# Entrenar y evaluar cada idioma por separado
for lang in languages:
    train_and_evaluate_language(lang)


Mounted at /content/drive
Inicializando el procesador...
Iniciando entrenamiento para italian...
Archivo .pt actualizado para italian: 1995 muestras.
Entrenando italian, Epoch 1/5


100%|██████████| 250/250 [47:02<00:00, 11.29s/it]


Loss promedio para italian, Epoch 1: 0.2120
Entrenando italian, Epoch 2/5


100%|██████████| 250/250 [31:24<00:00,  7.54s/it]


Loss promedio para italian, Epoch 2: 0.0598
Entrenando italian, Epoch 3/5


...

Loss promedio para swahili, Epoch 4: 0.0092
Entrenando swahili, Epoch 5/5


100%|██████████| 250/250 [35:13<00:00,  8.45s/it]


Loss promedio para swahili, Epoch 5: 0.0085


100%|██████████| 250/250 [24:35<00:00,  5.90s/it]

WER para swahili: 0.2570, CER para swahili: 0.2351


In [ ]:
import os
import warnings
from google.colab import drive
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from jiwer import wer, cer
from torch.cuda.amp import autocast, GradScaler

# Suprimir warnings
warnings.filterwarnings("ignore")

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas en Google Drive o local para cada idioma
audio_dir = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/wavpreprocessed",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/wavpreprocessed",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/wavpreprocessed"
}

pt_dir = {
    "italian": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Italian/preprocessed_transcriptionsit.pt",
    "spanish": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Spanish/preprocessed_transcriptionses.pt",
    "swahili": "/content/drive/My Drive/TinyML for Translation/SST & TTS/Swahili/preprocessed_transcriptionssw.pt"
}

# Inicializar el procesador
print("Inicializando el procesador...")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# Añadir tokens de idioma al tokenizador
language_tokens = {"spanish": "<|es|>", "italian": "<|it|>", "swahili": "<|sw|>"}
processor.tokenizer.add_tokens(list(language_tokens.values()))

# Inicializar el modelo y ajustar el tamaño de los embeddings
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model.resize_token_embeddings(len(processor.tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Función para sincronizar y actualizar los datos preprocesados
def update_preprocessed_data(language):
    with open(pt_dir[language], 'rb') as f:
        data = torch.load(f)
    filtered_data = []
    audio_directory = audio_dir[language]
    for item in data:
        audio_file = os.path.join(audio_directory, item["audio_file"])
        if os.path.isfile(audio_file):
            filtered_data.append(item)
    torch.save(filtered_data, pt_dir[language])
    print(f"Archivo .pt actualizado para {language}: {len(filtered_data)} muestras.")
    return filtered_data

# Función para cargar y preparar datos multilingües
def load_and_prepare_multilingual_data(languages):
    preprocessed_data = []
    for language in languages:
        data = update_preprocessed_data(language)
        for item in data:
            item['language'] = language
            item['language_code'] = language_tokens[language].strip('<|>')
            preprocessed_data.append(item)
    return preprocessed_data

# Dataset personalizado que incluye tokens de idioma en las etiquetas
class AudioTextDataset(Dataset):
    def __init__(self, audio_dirs, preprocessed_data):
        self.audio_dirs = audio_dirs
        self.preprocessed_data = preprocessed_data

    def __len__(self):
        return len(self.preprocessed_data)

    def __getitem__(self, idx):
        item = self.preprocessed_data[idx]
        language = item["language"]
        audio_file = os.path.join(self.audio_dirs[language], item["audio_file"])

        try:
            waveform, sample_rate = torchaudio.load(audio_file)
            if sample_rate != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
                waveform = resampler(waveform)
            input_features = processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt").input_features

            # Añadir el token de idioma al inicio de las etiquetas
            language_token = language_tokens[language]
            language_token_id = processor.tokenizer.convert_tokens_to_ids(language_token)
            labels = torch.cat([torch.tensor([language_token_id]), item["labels"]])

            return {
                "input_features": input_features.squeeze(0),
                "labels": labels
            }
        except Exception as e:
            print(f"Error al cargar el archivo de audio {audio_file}: {e}")
            return None

# Función collate_fn modificada
def collate_fn(batch):
    # Filtrar None
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    input_features = [item["input_features"] for item in batch]
    labels = [item["labels"] for item in batch]

    # Padding de input_features
    input_features_padded = torch.nn.utils.rnn.pad_sequence(input_features, batch_first=True, padding_value=0.0)

    # Padding de labels
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=processor.tokenizer.pad_token_id)

    return {
        "input_features": input_features_padded,
        "labels": labels_padded
    }

# Función de entrenamiento para modelo multilingüe con Gradient Accumulation
def train_multilingual(model, dataloader, epochs=5, lr=1e-4, gradient_accumulation_steps=4):
    optimizer = AdamW(model.parameters(), lr=lr)
    scaler = GradScaler()
    writer = SummaryWriter(log_dir='/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/tensorboard_logs/')
    global_step = 0
    save_interval = 1000  # Ajusta según tus necesidades

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        model.train()
        epoch_loss = 0
        for step, batch in enumerate(tqdm(dataloader)):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            # Verificar el tamaño del batch
            batch_size_current = input_features.size(0)
            if batch_size_current != 8:
                print(f"¡Advertencia! Tamaño del batch en el paso {step}: {batch_size_current}")

            with autocast():
                outputs = model(input_features=input_features, labels=labels)
                loss = outputs.loss / gradient_accumulation_steps  # Normalizar pérdida

            scaler.scale(loss).backward()

            if (step + 1) % gradient_accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                torch.cuda.empty_cache()  # Limpiar caché de la GPU

            epoch_loss += loss.item() * gradient_accumulation_steps
            writer.add_scalar("Loss/train", loss.item() * gradient_accumulation_steps, global_step)
            global_step += 1

            if global_step % save_interval == 0:
                model.save_pretrained(f'/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/checkpoint_step_{global_step}')

        avg_loss = epoch_loss / len(dataloader)
        print(f"Loss promedio para Epoch {epoch+1}: {avg_loss:.4f}")
        writer.add_scalar("Loss/epoch", avg_loss, epoch)

        model.save_pretrained(f'/content/drive/My Drive/TinyML for Translation/SST & TTS/Multilingual/exported_model/epoch_{epoch+1}')

    writer.close()

# Función de evaluación para modelo multilingüe
def evaluate_multilingual(model, dataloader):
    model.eval()
    references = []
    hypotheses = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            if batch is None:
                continue
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            generated_ids = model.generate(input_features)
            transcriptions = processor.batch_decode(generated_ids, skip_special_tokens=True)
            references_batch = processor.batch_decode(labels, skip_special_tokens=True)

            references.extend(references_batch)
            hypotheses.extend(transcriptions)

    total_wer = wer(references, hypotheses)
    total_cer = cer(references, hypotheses)

    print(f"WER: {total_wer:.4f}, CER: {total_cer:.4f}")
    return total_wer, total_cer

# Cargar y preparar los datos multilingües
languages = ["italian", "spanish", "swahili"]
preprocessed_data = load_and_prepare_multilingual_data(languages)
dataset = AudioTextDataset(audio_dir, preprocessed_data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=0)

# Entrenar el modelo multilingüe
train_multilingual(model, dataloader, epochs=5, lr=1e-4, gradient_accumulation_steps=4)

# Evaluar el modelo multilingüe
evaluate_multilingual(model, dataloader)


Mounted at /content/drive
Inicializando el procesador...
Archivo .pt actualizado para italian: 1995 muestras.
Archivo .pt actualizado para spanish: 2000 muestras.
Archivo .pt actualizado para swahili: 2000 muestras.
Epoch 1/5


100%|█████████▉| 749/750 [3:20:24<00:16, 16.41s/it]

¡Advertencia! Tamaño del batch en el paso 749: 3


100%|██████████| 750/750 [3:20:31<00:00, 16.04s/it]


Loss promedio para Epoch 1: 0.2239


...

100%|██████████| 750/750 [1:56:55<00:00,  9.35s/it]
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}


Loss promedio para Epoch 5: 0.0120


  0%|          | 0/750 [00:00<?, ?it/s]Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 750/750 [1:27:06<00:00,  6.97s/it]


WER: 0.2596, CER: 0.2109


(0.25960073889230967, 0.21087400765391426)